Trying to create a personal 3D stitiching algorithm

In [58]:
import tifffile
import pandas as pd
from skimage.measure import regionprops, regionprops_table
import napari
import numpy as np
import os



def properties_mask(image):

    # Get the properties of every mask in the image
    props = regionprops(image)

    data = []

    # For every mask found, get the label, centeroid, boundingbox, and volume
    for prop in props:
        label = prop.label
        centroid = prop.centroid  # (z, y, x)
        bounding_box = prop.bbox  # (min_z, min_y, min_x, max_z, max_y, max_x)
        volume = prop.area
        
        # Pre-compute coordinate set for fast IOU calculations later
        coords_2d = prop.coords[:, 1:]
        coords_set = set(map(tuple, coords_2d))

        data.append(
            {
                "label": label,
                "z": centroid[0],
                "y": centroid[1],
                "x": centroid[2],
                "bounding_box": bounding_box,
                "volume": volume,
                "coords_set": coords_set,
            }
        )

    # Make a pandas data frame from the data
    df_props = pd.DataFrame(data)

    return df_props


def properties_channel(mask, image, type=None):

    # Get the properties of the mask for the channel you specified (image)
    props_channel = regionprops_table(
        mask, intensity_image=image, properties=["label", "mean_intensity"]
    )

    # Make this into a pandas dataframe and rename some variables to the name of the channel for later merging
    props_channel = pd.DataFrame(props_channel).rename(
        columns={
            "label": "label",
            "mean_intensity": f"raw_{type}",
        }
    )

    return props_channel

def make_unique_mask(mask):
    """Makes a fresh mask from a already stitched mask"""
    unique_mask = []
    number = 1
    for z in range(mask.shape[0]):
        # go over every unique number in the mask and create a new slice with a new unique number 1 higher
        slice = mask[z]
        unique_slice = mask[z].copy()
        for val in range(1, slice.max() + 1):
            unique_slice[slice == val] = number
            number += 1
        unique_mask.append(unique_slice)
    unique_mask = np.stack(unique_mask, axis=0)

    return unique_mask

Dual nuclei image

In [59]:
# generate mask
import os
import sys
sys.path.insert(0, os.path.dirname(os.getcwd()))
import utils

image = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\cropped\Frame-0.tif"
)

nuclei = image[:, 0:1]

nuclei = np.max(image, axis=1)

model = utils.load_model(r"C:\Users\6331823\Local SSD\Train dual nuclei 20x\models\dual_nuclei")

mask = utils.segment(nuclei, model)

tifffile.imwrite(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\Frame-0_segmented.tif",
    mask,
)

KeyboardInterrupt: 

In [ ]:
image = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\cropped\Frame-0.tif"
)
crc_image = image[:,0]
wt_image = image[:,1]
mask = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\Frame-0_segmented.tif"
)

print(crc_image.shape, wt_image.shape, mask.shape)

unique_mask = make_unique_mask(mask)

props_mask = properties_mask(unique_mask)
props_wt = properties_channel(unique_mask, wt_image, type="wt")
props_crc = properties_channel(unique_mask, crc_image, type="crc")

# Merge all the dataframes together on the label
df_final = props_mask.merge(props_wt, on="label").merge(props_crc, on="label")

(63, 356, 370) (63, 356, 370) (63, 356, 370)


In [87]:
# using the fake data
import os
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))
import utils

output_directory = r"C:\Users\6331823\Local SSD\Fake_data"

image = tifffile.imread(os.path.join(output_directory, "realistic_organoid.tif"))

model = utils.load_model(
    r"C:\Users\6331823\Local SSD\Train dual nuclei 20x\models\dual_nuclei"
)

mask = utils.segment(image, model)

tifffile.imwrite(
    os.path.join(output_directory, "realistic_organoid_segmented.tif"),
    mask,
)

Using CUDA for processing.
loaded custom model: dual_nuclei


In [109]:
image = tifffile.imread(
    os.path.join(output_directory, "realistic_organoid.tif"))

image_wt = image_crc = image
# make a fake channel layer at axis 1
mask = tifffile.imread(
    os.path.join(output_directory, "realistic_organoid_segmented.tif")
)
perfect_mask = tifffile.imread(
    os.path.join(output_directory, "perfect_mask.tif")
)

unique_mask = make_unique_mask(mask)

props_mask = properties_mask(unique_mask)
props_wt = properties_channel(unique_mask, image, type="wt")
props_crc = properties_channel(unique_mask, image, type="crc")

# Merge all the dataframes together on the label
df_final = props_mask.merge(props_wt, on="label").merge(props_crc, on="label")

In [ ]:
def get_mask_properties(mask, image1, image2 = None):
    unique_mask = make_unique_mask(mask)

    props_mask = properties_mask(unique_mask)
    props_wt = properties_channel(unique_mask, image1, type="wt")
    if image2 is not None:
        props_crc = properties_channel(unique_mask, image2, type="crc")
        # Merge all the dataframes together on the label
        df_final = props_mask.merge(props_wt, on="label").merge(props_crc, on="label")
    else:
        df_final = props_mask.merge(props_wt, on="label")

    return df_final, unique_mask

def calculate_iou(coords_sets, idx1, idx2):
    """
    Calculate overlap metric for cell matching.
    Returns 1 if the smaller cell is completely contained in the larger cell.
    Otherwise returns intersection over union (standard IOU).
    """
    set1 = coords_sets[idx1]
    set2 = coords_sets[idx2]

    # Check if either set is empty
    len1, len2 = len(set1), len(set2)
    if len2 == 0:
        return 0

    # Calculate intersection (overlapping pixels)
    intersection = len(set1 & set2)

    if intersection == 0:
        return 0  # No overlap = different cells

    # If the smaller cell is completely contained in the larger cell, return 1
    min_len = min(len1, len2)
    if intersection == min_len:
        return 1.0  # One cell completely covers the other

    # Otherwise, use standard IOU
    union = len1 + len2 - intersection
    return intersection / union

def IOU_stitching(data, unique_mask):

    print("Starting IOU-based stitching...")
    df_final = data.copy()
    # Reset index to ensure sequential 0-based indexing
    df_final = df_final.reset_index(drop=True)

    df_final["3d_label"] = -1  # Use -1 to indicate unassigned
    mask_3D = np.zeros_like(unique_mask)
    coords_sets = df_final["coords_set"].to_dict()

    z_groups = {}
    for idx, z_val in df_final["z"].items():
        z_int = int(round(z_val))
        if z_int not in z_groups:
            z_groups[z_int] = []
        z_groups[z_int].append(idx)

    z_array = df_final["z"].values
    x_array = df_final["x"].values
    y_array = df_final["y"].values
    labels_array = np.full(len(df_final), -1, dtype=np.int32)

    def stitch_forward(idx, current_label):
        """Simplified recursive stitching - idx is now both pandas index and array position"""
        current_z = int(round(z_array[idx]))

        # Assign the current label
        labels_array[idx] = current_label

        # Fast lookup: get only indices in the next slice
        next_z = current_z + 1
        if next_z not in z_groups:
            return

        candidates = z_groups[next_z]

        # Filter unassigned candidates
        unassigned_candidates = [c for c in candidates if labels_array[c] == -1]

        if len(unassigned_candidates) == 0:
            return

        # Vectorized distance calculation
        unassigned_array = np.array(unassigned_candidates)
        distances = np.sqrt(
            (x_array[unassigned_array] - x_array[idx]) ** 2
            + (y_array[unassigned_array] - y_array[idx]) ** 2
        )

        # Find the closest candidate
        closest_idx = np.argmin(distances)
        closest_distance = distances[closest_idx]

        # Only proceed if the closest cell is within threshold
        if closest_distance > 10:
            return

        # Get the closest candidate's index
        closest_candidate = unassigned_array[closest_idx]

        # Check IOU with the closest candidate only
        iou = calculate_iou(coords_sets,idx, closest_candidate)
        if iou > 0.3:
            # Recursively stitch this cell forward
            stitch_forward(closest_candidate, current_label)

    unique_label = 0
    for idx in range(len(df_final)):
        if labels_array[idx] == -1:
            unique_label += 1
            stitch_forward(idx, unique_label)

    # Copy results back to dataframe
    df_final["3d_label"] = labels_array

    print(f"Total unique 3D cells found: {unique_label} via IOU stitching")

    # Vectorized mask creation
    mask_3D = np.zeros_like(unique_mask)
    for idx, row in df_final.iterrows():
        z = int(row["z"])
        label_3d = labels_array[idx]
        mask_3D[z][unique_mask[z] == row["label"]] = label_3d

    return df_final, mask_3D


def break_score(values, i):
    # Calculate intensity-based anomalies by getting a score that resembles high-low-high intensity value patterns
    # Calculate ratio between slice i and below i, and i and above i
    values_bottom = max(values[:i])
    values_top = max(values[i + 1 :])
    ratio_1 = values_bottom / values[i]
    ratio_2 = values[i] / values_top

    # divide this again, now high low high gets higher score then other patterns
    ratio_total = ratio_1 / ratio_2
    return ratio_total


def point_line_distance(z_positions, x_positions, y_positions, points=3):
    # Fit line to specified points (int, tuple, or slice)
    if isinstance(points, tuple):
        points = slice(*points)
    elif isinstance(points, int):
        points = slice(None, points)

    z_fit = np.array(z_positions[points])
    x_fit = np.array(x_positions[points])
    y_fit = np.array(y_positions[points])

    # Calculate line parameters
    A = np.vstack([z_fit, np.ones(len(z_fit))]).T
    m_x, c_x = np.linalg.lstsq(A, x_fit, rcond=None)[0]
    m_y, c_y = np.linalg.lstsq(A, y_fit, rcond=None)[0]

    # Create line function: given z, returns (x, y)
    line_function = lambda z: (m_x * z + c_x, m_y * z + c_y)

    # Calculate distances for all points
    distances = []
    for i in range(len(z_positions)):
        z_val = z_positions[i]
        x_line = m_x * z_val + c_x
        y_line = m_y * z_val + c_y
        distance = np.sqrt(
            (x_positions[i] - x_line) ** 2 + (y_positions[i] - y_line) ** 2
        )
        distances.append(distance)
    return distances, line_function


def slope_change(distances):
    # First derivative (slope between consecutive points)
    first_deriv = np.diff(distances)

    # Second derivative (change in slope)
    second_deriv = np.diff(first_deriv)

    # Pad to match original length
    second_deriv_padded = np.concatenate([[0], second_deriv, [0]])

    return second_deriv_padded


# Create a new dataframe grouped by 3D label
def calculate_3d_properties(df_3d_props):
    # Optional: add number of slices per 3D cell
    raw_cols = [c for c in df_3d_props.columns if c.startswith("raw_")]

    df_3d_props["num_slices"] = df_3d_props["labels_2d"].apply(len)
    init_arrays = lambda n: [0] * n
    for col in [
        "volumes_break_score",
        "final_break_score",
        "break_score_up",
        "break_score_down",
        "intensity_break_score",
    ]:
        df_3d_props[col] = df_3d_props["num_slices"].apply(init_arrays)

    if raw_cols:

        def max_intensity(row):
            per_channel = [row[col] for col in raw_cols]
            return [max(values) for values in zip(*per_channel)]

    else:

        def max_intensity(row):
            return [0] * row["num_slices"]

    df_3d_props["nuclei_intensity"] = df_3d_props.apply(max_intensity, axis=1)

    # itterate over every row, look at the volumes, find cells where the pattern is big small big
    for idx, row in df_3d_props.iterrows():
        volumes = row["volumes"]
        intensities = row["nuclei_intensity"]
        if len(volumes) < 5:
            continue  # At least 5 slices to break something

        for i in range(1, len(volumes) - 1):
            df_3d_props.at[idx, "volumes_break_score"][i] = break_score(volumes, i)
            df_3d_props.at[idx, "intensity_break_score"][i] = break_score(
                intensities, i
            )
            # multiply both scores to get a final score

        # Calculate a line based on first 3 points and calculate distance from this line for other points
        z_positions = row["z_positions"]
        x_positions = row["x_centroids"]
        y_positions = row["y_centroids"]
        line_distance_up, _ = point_line_distance(z_positions, x_positions, y_positions)
        # # get delta between consecutive points
        # delta_line_distance_up = abs(np.diff(line_distance_up))
        # break_score_up = [0] + [score / 1 for score in delta_line_distance_up]
        break_score_up = abs(slope_change(line_distance_up))
        df_3d_props.at[idx, "break_score_up"] = break_score_up

        # Calculate a line based on last 3 points and calculate distance from this line for other points
        line_distance_down, _ = point_line_distance(
            z_positions[::-1], x_positions[::-1], y_positions[::-1]
        )
        line_distance_down = line_distance_down[::-1]
        # delta_line_distance_down = abs(np.diff(line_distance_down))
        # break_score_down = [0] + [score / 1 for score in delta_line_distance_down]
        break_score_down = abs(slope_change(line_distance_down))
        df_3d_props.at[idx, "break_score_down"] = break_score_down

        # calculate final break score by multiplying all scores
        final_scores = []
        for v_score, i_score, up_score, down_score in zip(
            df_3d_props.at[idx, "volumes_break_score"],
            df_3d_props.at[idx, "intensity_break_score"],
            df_3d_props.at[idx, "break_score_up"],
            df_3d_props.at[idx, "break_score_down"],
        ):
            final_score = np.max([up_score, down_score]) * v_score * i_score
            final_scores.append(final_score)
        df_3d_props.at[idx, "final_break_score"] = final_scores

    return df_3d_props


def make_data_frame_3d(data):
    raw_cols = [c for c in data.columns if c.startswith("raw_")]

    agg_map = {
        "label": list,
        "z": list,
        "y": list,
        "x": list,
        "volume": list,
    }
    agg_map.update({col: list for col in raw_cols})

    data = data.groupby("3d_label").agg(agg_map).reset_index()

    rename_map = {
        "label": "labels_2d",
        "z": "z_positions",
        "y": "y_centroids",
        "x": "x_centroids",
        "volume": "volumes",
    }
    data = data.rename(columns=rename_map)

    return data


def stitch_loop(df_final, threshold=2.5):
    # Make 3D dataframe from current df_final state
    uncut_data = make_data_frame_3d(df_final)

    # Calculate break scores
    df_3d_props = calculate_3d_properties(uncut_data)

    # # This code can show how it is cutting during debugging
    # # Drop num_slices
    # df_3d_props_long = df_3d_props.drop(columns=["num_slices"], errors="ignore")

    # # Add slice index to keep alignment after explode
    # df_3d_props_long["slice_idx"] = df_3d_props_long["volumes"].apply(lambda v: list(range(len(v))))

    # # Identify list-like columns (excluding the key column)
    # list_cols = [
    #     c for c in df_3d_props_long.columns
    #     if c != "3d_label" and df_3d_props_long[c].apply(lambda v: isinstance(v, (list, tuple, np.ndarray))).all()
    # ]

    # # Explode all list-like columns together (keeps rows aligned)
    # df_3d_props_long = df_3d_props_long.explode(list_cols, ignore_index=True)
    # display(df_3d_props_long[df_3d_props_long["3d_label"] == 409])
    # Get the maximum existing 3D label to continue numbering
    new_label_counter = df_3d_props["3d_label"].max() + 1

    breaks_found = 0  # Track if any breaks were made

    cut_cells = []
    # Break cells at slices with break score above threshold
    for idx, row in df_3d_props.iterrows():
        break_scores = row["final_break_score"]
        labels_2d = row["labels_2d"]

        # Only consider breaks in middle slices (skip first 3 and last 3)
        # Need at least 3 slices on each side for valid detection
        if len(labels_2d) < 7:  # Too short to have valid breaks
            continue

        # Mask out first 3 and last 3 positions
        masked_scores = np.array(break_scores, dtype=float)
        masked_scores[:3] = 0  # First 3 slices get score 0
        masked_scores[-3:] = 0  # Last 3 slices get score 0

        # If no breaks above threshold, skip this cell
        if max(masked_scores) < threshold:
            continue

        breaks_found += 1

        # Find slice with highest break score
        max_break = max(masked_scores)
        break_position = [
            i for i, score in enumerate(masked_scores) if score == max_break
        ][0]

        # Get the 2D labels
        labels_2d = row["labels_2d"]

        # All slices after and including break get a new 3D label
        for i in range(break_position, len(labels_2d)):
            label_2d = labels_2d[i]
            df_final.loc[df_final["label"] == label_2d, "3d_label"] = new_label_counter

        cut_cells.append(row["3d_label"])
        cut_cells.append(new_label_counter)
        new_label_counter += 1

    return df_final, breaks_found, cut_cells

def stitch_3D(image1, image2=None, mask = None, threshold = 2.5):
    if mask is None:
        mask = utils.segment(image1, model)

    if image2 is not None:
        df_final, unique_mask = get_mask_properties(mask, image1, image2)
    else:
        df_final, unique_mask = get_mask_properties(mask, image1)
    # Initial IOU-based stitching
    IOU_stitched_df, IOU_stitched_mask = IOU_stitching(df_final, unique_mask)
    stitched_df = IOU_stitched_df.copy()

    # Iterative break and restitching
    max_iterations = 10
    iteration = 0
    all_cut_cells = []
    while iteration < max_iterations:
            iteration += 1
        print(f"Stitching iteration {iteration}...")
        stitched_df, breaks_found, cut_cells = stitch_loop(stitched_df, threshold=threshold)
        all_cut_cells.extend(cut_cells)
        print(f"Breaks made this iteration: {breaks_found}")

        if breaks_found == 0:
            print("No more breaks found. Stitching complete.")
            break

    # Final unique mask creation
    mask_3d_fixed = np.zeros_like(unique_mask)
    for idx, row in stitched_df.iterrows():
        z = int(row["z"])
        label_3d = row["3d_label"]
        mask_3d_fixed[z][unique_mask[z] == row["label"]] = label_3d

    print(f"Total unique 3D cells found: {stitched_df['3d_label'].nunique()} after stitching")

    return mask_3d_fixed, IOU_stitched_mask, stitched_df, IOU_stitched_df, all_cut_cells

In [ ]:
import os
output_directory = r"C:\Users\6331823\Local SSD\Fake_data"
# fake data stitching
image = tifffile.imread(os.path.join(output_directory, "realistic_organoid.tif"))
perfect_organoid = tifffile.imread(os.path.join(output_directory, "perfect_organoid.tif"))

image_wt = image_crc = image
# make a fake channel layer at axis 1
mask = tifffile.imread(
    os.path.join(output_directory, "realistic_organoid_segmented.tif")
)
perfect_mask = tifffile.imread(os.path.join(output_directory, "perfect_mask.tif"))

mask_3d_fixed, IOU_stitched_mask, stitched_df, IOU_stitched_df, all_cut_cells = stitch_3D(image_wt, image_crc, mask=mask, threshold=2.5)
print(f"actual amount of cells: {len(np.unique(perfect_mask)) - 1}")

# create a mask that only has cells that were cut
cut_cells_mask = np.zeros_like(mask_3d_fixed)
for cell_label in all_cut_cells:
    cut_cells_mask[mask_3d_fixed == cell_label] = cell_label

new_labels = np.zeros_like(mask_3d_fixed)
for idx, row in new_df.iterrows():
    z = int(row["z"])
    label_3d = row["3d_label"]
    coords_array = np.array(list(row["coords_set"]))

    if len(coords_array) > 0:
        y_coords = coords_array[:, 0].astype(int)
        x_coords = coords_array[:, 1].astype(int)
        new_labels[z, y_coords, x_coords] = label_3d
viewer = napari.Viewer()
scale = (2.5, 0.64, 0.64)
viewer.add_image(image, scale=scale, name="image", visible=False)
viewer.add_image(perfect_organoid, name="perfect organoid", scale=scale, visible=False)
viewer.add_labels(perfect_mask, name="perfect mask", scale=scale, visible=False)
viewer.add_labels(mask_3d_fixed, name="Final stitched mask", scale=scale)
viewer.add_labels(IOU_stitched_mask, name="IOU stitched mask", scale=scale, visible=False)
viewer.add_labels(cut_cells_mask, name="cut cells", scale=scale, visible=False)
viewer.add_labels(test, name = 'test', scale=scale, visible=False)
viewer.add_labels(new_labels, name="new_labels", scale=scale, visible=False)

Starting IOU-based stitching...
Total unique 3D cells found: 178 via IOU stitching
Stitching iteration 1...
Breaks made this iteration: 20
Stitching iteration 2...
Breaks made this iteration: 11
Stitching iteration 3...
Breaks made this iteration: 1
Stitching iteration 4...
Breaks made this iteration: 0
No more breaks found. Stitching complete.
Total unique 3D cells found: 210 after stitching
actual amount of cells: 200


<Labels layer 'new_labels' at 0x14125c82650>

In [56]:
test.shape

(77, 300, 300)

In [78]:
# For all cut cells, find if which other cut cells they are touching in 3D space
props = regionprops(cut_cells_mask)
new_df = stitched_df.copy()

properties = []
for prop in props:
    label = prop.label
    coords_3d = prop.coords # (z, y, x)
    # filter only coords that are highest and lowest z positions
    selected_coords_3d = coords_3d[
        (coords_3d[:, 0] == coords_3d[:, 0].min()) | (coords_3d[:, 0] == coords_3d[:, 0].max())
    ]   
    centeroid = prop.centroid
    properties.append({
    "label": label,
    "selected_coords_3d": selected_coords_3d,
    "coords_3d": coords_3d,
    "centeroid": centeroid
    })
properties = pd.DataFrame(properties)

properties["touching_cells"] = None

neighbor_offsets = [(-1, 0, 0), (1, 0, 0)]

for idx, cell in properties.iterrows():
    centeroid = cell["centeroid"]
    properties["distance"] = properties["centeroid"].apply(
        lambda x: np.sqrt(
            (x[0] - centeroid[0]) ** 2
            + (x[1] - centeroid[1]) ** 2
            + (x[2] - centeroid[2]) ** 2
        )
    )
    closest_cells = properties.sort_values("distance").iloc[1:6]

    touching_cells = []
    cell_coords_set = set(map(tuple, cell["selected_coords_3d"]))
    for _, other_cell in closest_cells.iterrows():
        other_coords = set(map(tuple, other_cell["coords_3d"]))
        if any(
            (z + dz, y, x) in other_coords
            for z, y, x in cell_coords_set
            for dz, _, _ in neighbor_offsets
        ):
            touching_cells.append(other_cell["label"])

    properties.at[idx, "touching_cells"] = touching_cells

# remove distance column
properties = properties.drop(columns=["distance"])
properties
# get a new dataframe with all combinations of touching cells
touching_pairs = []
for idx, cell in properties.iterrows():
    label = cell["label"]
    for touching_label in cell["touching_cells"]:
        if label < touching_label:  # Avoid duplicates
            cell_1_coords = cell["coords_3d"]
            cell_2_coords = properties[properties["label"] == touching_label]["coords_3d"].values[0]
            cell_1_centeroid = cell["centeroid"]
            cell_2_centeroid = properties[properties["label"] == touching_label]["centeroid"].values[0]
            if cell_1_centeroid[0] < cell_2_centeroid[0]:
                touching_pairs.append({"cell_1": label, "cell_2": touching_label,
                                    "cell_1_coords": cell_1_coords,
                                    "cell_2_coords": cell_2_coords,
                                    "cell_1_centeroid": cell_1_centeroid,
                                    "cell_2_centeroid": cell_2_centeroid,
                                    })
            else:
                touching_pairs.append({"cell_1": touching_label, "cell_2": label,
                                    "cell_1_coords": cell_2_coords,
                                    "cell_2_coords": cell_1_coords,
                                    "cell_1_centeroid": cell_2_centeroid,
                                    "cell_2_centeroid": cell_1_centeroid,
                                    })


def slice_centroids_df(coords_3d):
    df = pd.DataFrame(coords_3d, columns=["z", "y", "x"])
    out = (
        df.groupby("z")[["y", "x"]]
        .mean()
        .reset_index()
        .assign(y=lambda d: d["y"].astype(float), x=lambda d: d["x"].astype(float))
    )
    return list(out.itertuples(index=False, name=None))

def check_split_cells(centroids_2d, pred_point_1, pred_point_2, pair):
    last_z = centroids_2d[-1][0]
    cell_1_last_slice = np.array(
        [coord for coord in pair["cell_1_coords"] if coord[0] == last_z]
    )
    original_centroid = centroids_2d[-1]
    original_distance_1 = np.sqrt(
        (original_centroid[1] - y_last_slice_line_1)**2 + 
        (original_centroid[2] - x_last_slice_line_1)**2
    )

    # Create splitting line perpendicular to the line between the two predicted points
    # Direction vector between predicted points
    direction = pred_point_2 - pred_point_1
    # Perpendicular vector (rotate 90 degrees)
    normal = direction / (np.linalg.norm(direction) + 1e-8)
    # Midpoint between predicted points
    midpoint = (pred_point_1 + pred_point_2) / 2

    part_1_coords = []
    part_2_coords = []

    for coord in cell_1_last_slice:
        z, y, x = coord
        point = np.array([y, x])
        # Vector from midpoint to this point
        to_point = point - midpoint
        # Dot product determines which side of the line
        side = np.dot(to_point, normal)

        if side <= 0:
            part_1_coords.append(coord)
        else:
            part_2_coords.append(coord)

    min_size = max(
        3, len(cell_1_last_slice) * 0.1
    )  # At least 10% of pixels or 3 pixels

    if len(part_1_coords) < min_size or len(part_2_coords) < min_size:
        print(f"  Warning: Split too unbalanced ({len(part_1_coords)} vs {len(part_2_coords)} pixels)")
        return None

    # Calculate new centroids
    part_1_coords = np.array(part_1_coords)
    part_2_coords = np.array(part_2_coords)

    centroid_1 =  np.mean(part_1_coords[:, 1:], axis=0)  # (y, x)
    centroid_2 = np.mean(part_2_coords[:, 1:], axis=0)  # (y, x)

    # Calculate distances from new centroids to predicted points
    dist_c1_to_pred1 = np.sqrt(
        (centroid_1[0] - pred_point_1[0]) ** 2
        + (centroid_1[1] - pred_point_1[1]) ** 2
    )
    dist_c2_to_pred2 = np.sqrt(
        (centroid_2[0] - pred_point_2[0]) ** 2
        + (centroid_2[1] - pred_point_2[1]) ** 2
    )

    # Check if splitting improves the fit
    avg_new_distance = (dist_c1_to_pred1 + dist_c2_to_pred2) / 2

    print(f"  Cell {pair['cell_1']} & {pair['cell_2']}:")
    if avg_new_distance < original_distance_1:
        print(
            f"    ✓ Splitting improves fit! (reduction: {original_distance_1 - avg_new_distance:.2f})"
        )
        return(part_1_coords, part_2_coords)
    else:
        print(f"    ✗ Splitting does not improve fit")
        return None

for pair in touching_pairs:
    centroids_2d_1 = slice_centroids_df(pair["cell_1_coords"])
    centroids_2d_2 = slice_centroids_df(pair["cell_2_coords"])

    distances_1, line_1 = point_line_distance(
        [c[0] for c in centroids_2d_1],  # z
        [c[2] for c in centroids_2d_1],  # x
        [c[1] for c in centroids_2d_1],  # y
        points=(0, -1),
    )
    distances_2, line_2 = point_line_distance(
        [c[0] for c in centroids_2d_2][::-1],  # z
        [c[2] for c in centroids_2d_2][::-1],  # x
        [c[1] for c in centroids_2d_2][::-1],  # y
        points=(0, -1),
    )
    x_last_slice_line_1, y_last_slice_line_1 = line_1(centroids_2d_1[-1][0])
    x_last_slice_line_2, y_last_slice_line_2 = line_2(centroids_2d_2[-1][0])
    # if the predicted point is inside the cell_1 coords, print("Predicted point is inside cell 1")
    cell_1_coords_set = set(map(tuple, pair["cell_1_coords"]))
    cell_2_coords_set = set(map(tuple, pair["cell_2_coords"]))

    # Check if both predicted points are inside the overlapping slice
    pred_in_cell_1_line_1 = (
        centroids_2d_1[-1][0],
        int(round(y_last_slice_line_1)),
        int(round(x_last_slice_line_1)),
    ) in cell_1_coords_set
    # print(int(round(y_last_slice_line_1)), int(round(x_last_slice_line_1)))
    # print(pair["cell_1_coords"][centroids_2d_1[-1][0] == pair["cell_1_coords"][:,0]])
    pred_in_cell_1_line_2 = (
        centroids_2d_1[-1][0],
        int(round(y_last_slice_line_2)),
        int(round(x_last_slice_line_2)),
    ) in cell_1_coords_set

    if pred_in_cell_1_line_1 and pred_in_cell_1_line_2:
        # Check minimum distance between predicted points
        pred_point_1 = np.array([y_last_slice_line_1, x_last_slice_line_1])
        pred_point_2 = np.array([y_last_slice_line_2, x_last_slice_line_2])
        distance_between_preds = np.linalg.norm(pred_point_2 - pred_point_1)

        if distance_between_preds < 1:  # Minimum 1 pixel apart
            continue

        # split the cell 1 coords of this last z slice into two parts, perpendicular to the line between the two predicted points
        # Get the last z slice coords for both cells

        result = check_split_cells(
            centroids_2d_1, pred_point_1, pred_point_2, pair
        )
        if result is None:
            continue
        part_1_coords, part_2_coords = result
        # Update the new_df to reflect the split
        last_z = centroids_2d_1[-1][0]

        # Convert part coords to (y, x) tuples for coords_set
        part_1_coords_2d = set(map(tuple, part_1_coords[:, 1:]))  # (y, x)
        part_2_coords_2d = set(map(tuple, part_2_coords[:, 1:]))  # (y, x)

        # Find the row(s) for cell_1 at last_z and update coords_set
        cell_1_mask = (new_df["3d_label"] == pair["cell_1"]) & (new_df["z"] == last_z)
        if cell_1_mask.any():
            # Overwrite coords_set for cell_1 at last z slice - assign to each row
            for idx in new_df[cell_1_mask].index:
                new_df.at[idx, "coords_set"] = part_1_coords_2d

        # Add a new row for cell_2 at last z slice
        new_row = new_df[cell_1_mask].iloc[0].copy()
        new_row["3d_label"] = pair["cell_2"]
        new_row["coords_set"] = part_2_coords_2d
        new_row["volume"] = len(part_2_coords_2d)
        new_df = pd.concat([new_df, pd.DataFrame([new_row])], ignore_index=True)

  Cell 68 & 199:
    ✗ Splitting does not improve fit
  Cell 69 & 180:
    ✗ Splitting does not improve fit
  Cell 71 & 181:
    ✓ Splitting improves fit! (reduction: 1.38)
  Cell 72 & 200:
    ✗ Splitting does not improve fit
  Cell 76 & 201:
    ✗ Splitting does not improve fit
  Cell 78 & 185:
    ✓ Splitting improves fit! (reduction: 4.51)
  Cell 79 & 202:
    ✗ Splitting does not improve fit
  Cell 80 & 187:
    ✗ Splitting does not improve fit
  Cell 83 & 188:
    ✓ Splitting improves fit! (reduction: 0.27)
  Cell 86 & 189:
    ✗ Splitting does not improve fit
  Cell 87 & 190:
    ✗ Splitting does not improve fit
  Cell 89 & 203:
    ✓ Splitting improves fit! (reduction: 6.18)
  Cell 91 & 192:
    ✓ Splitting improves fit! (reduction: 3.37)
  Cell 93 & 193:
    ✗ Splitting does not improve fit
  Cell 108 & 198:
    ✓ Splitting improves fit! (reduction: 0.02)
  Cell 210 & 179:
    ✗ Splitting does not improve fit
  Cell 200 & 182:
    ✓ Splitting improves fit! (reduction: 1.80)
  

In [86]:
new_df[new_df["3d_label"] == 181]

,label,z,y,x,bounding_box,volume,coords_set,raw_wt,raw_crc,3d_label
830,831,34.0,31.991453,106.880342,"(34, 28, 98, 35, 37, 117)",117.0,"{(32, 101), (33, 100), (32, 110), (30, 110), (...",37.991453,37.991453,181
854,855,35.0,32.020408,106.938776,"(35, 27, 97, 36, 38, 118)",147.0,"{(32, 101), (33, 100), (27, 114), (35, 97), (3...",57.782313,57.782313,181
881,882,36.0,32.071006,106.804734,"(36, 26, 96, 37, 39, 119)",169.0,"{(32, 101), (33, 100), (27, 114), (35, 97), (3...",76.118343,76.118343,181
905,906,37.0,32.053254,106.863905,"(37, 26, 96, 38, 39, 119)",169.0,"{(32, 101), (33, 100), (27, 114), (35, 97), (3...",91.497041,91.497041,181
930,931,38.0,32.017647,106.841176,"(38, 26, 96, 39, 39, 119)",170.0,"{(32, 101), (33, 100), (27, 114), (35, 97), (3...",94.417647,94.417647,181
954,955,39.0,31.888889,107.024691,"(39, 26, 96, 40, 38, 119)",162.0,"{(32, 101), (33, 100), (27, 114), (35, 97), (3...",87.111111,87.111111,181
980,981,40.0,31.794702,107.006623,"(40, 26, 96, 41, 38, 118)",151.0,"{(32, 101), (33, 100), (27, 114), (35, 97), (3...",69.397351,69.397351,181
1006,1007,41.0,31.626866,107.567164,"(41, 27, 98, 42, 37, 118)",134.0,"{(32, 101), (33, 100), (27, 114), (32, 110), (...",48.283582,48.283582,181
1032,1033,42.0,31.489796,109.551020,"(42, 28, 100, 43, 36, 119)",98.0,"{(33, 100), (32, 110), (30, 110), (33, 109), (...",31.989796,31.989796,181
1864,808,33.0,33.415385,108.692308,"(33, 29, 100, 34, 39, 118)",74.0,"{(32, 101), (30, 107), (30, 113), (32, 104), (...",24.761538,24.761538,181


In [5]:
cut_cells_data = stitched_df[stitched_df["3d_label"].isin(all_cut_cells)]
simple_data = cut_cells_data[["3d_label", "z", "y", "x"]].copy()
summary_3d = []
neighbor_offsets = [(-1, 0, 0), (1, 0, 0)]

for cell in all_cut_cells:
    cell_data = cut_cells_data[cut_cells_data["3d_label"] == cell]
    centeroid = cell_data[["z", "y", "x"]].mean().values
    # Flatten all coords_set into a single set, adding z coordinate from each row
    coords_3d_flat = set()
    for idx, row in cell_data.iterrows():
        z = int(round(row["z"]))
        for y, x in row["coords_set"]:
            coords_3d_flat.add((z, y, x))

    summary_3d.append(
        {
            "3d_label": cell,
            "num_slices": len(cell_data),
            "centeroid_z": centeroid[0],
            "centeroid_y": centeroid[1],
            "centeroid_x": centeroid[2],
            "coords_3d": coords_3d_flat,  # Now stores a single set, not a list of sets
        }
    )
summary_3d = pd.DataFrame(summary_3d)
summary_3d["touching_cells"] = None

for cell in all_cut_cells:
    cell_data = summary_3d[summary_3d["3d_label"] == cell]
    # calculate distance to all other cells in summary_3d
    centeroid = cell_data[["centeroid_z", "centeroid_y", "centeroid_x"]].values[0]
    summary_3d["distance"] = summary_3d.apply(
        lambda row: np.sqrt(
            (row["centeroid_z"] - centeroid[0]) ** 2
            + (row["centeroid_y"] - centeroid[1]) ** 2
            + (row["centeroid_x"] - centeroid[2]) ** 2
        ),
        axis=1,
    )

    closest_cells = (
        summary_3d[summary_3d["3d_label"] != cell]
        .sort_values("distance")[1:6]
    )
    touching_cells = []
    cell_coords_set = cell_data["coords_3d"].values[0]  # Already a set

    for _, other_cell in closest_cells.iterrows():
        other_coords = other_cell["coords_3d"]  # Already a set
        if any(
            (z + dz, y, x) in other_coords
            for z, y, x in cell_coords_set
            for dz, _, _ in neighbor_offsets
        ):
            touching_cells.append(other_cell["3d_label"])
    # Find the index where 3d_label == cell and assign the value
    cell_idx = summary_3d[summary_3d["3d_label"] == cell].index[0]
    summary_3d.at[cell_idx, "touching_cells"] = touching_cells

# Remove the distance column
summary_3d = summary_3d.drop(columns=["distance"])
display(summary_3d.head())

,3d_label,num_slices,centeroid_z,centeroid_y,centeroid_x,coords_3d,touching_cells
0,68,7,26.0,198.961090,256.865027,"{(26, 203, 255), (28, 200, 261), (27, 199, 253...",[199]
1,179,4,40.5,188.234340,267.886611,"{(39, 182, 268), (39, 193, 268), (41, 182, 270...",[204]
2,69,9,28.0,115.270187,263.993250,"{(26, 117, 261), (26, 120, 266), (25, 116, 265...",[]
3,180,8,36.5,110.322110,268.207891,"{(39, 107, 265), (40, 111, 269), (35, 118, 273...",[108]
4,71,9,29.0,35.761089,111.494037,"{(31, 37, 105), (28, 33, 111), (26, 33, 118), ...",[]


In [40]:
# real data stitching
image = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\cropped\Frame-0.tif"
)
print(image.shape)
crc_image = image[:, 0]
print(crc_image.shape)
wt_image = image[:, 1]
mask = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\Frame-0_segmented.tif"
)

mask_3d_fixed, IOU_stitched_mask, stitched_df, IOU_stitched_df, _ = stitch_3D(
    wt_image, crc_image, mask=mask, threshold=2.5
)

viewer = napari.Viewer(ndisplay=3)
scale = (2.5, 0.64, 0.64)
viewer.add_image(wt_image, scale=scale, name="wt image", visible=False, blending="additive")
viewer.add_image(crc_image, scale=scale, name="crc image", visible=False, blending="additive")
viewer.add_labels(mask_3d_fixed, name="Final stitched mask", scale=scale)
viewer.add_labels(
    IOU_stitched_mask, name="IOU stitched mask", scale=scale, visible=False
)

(63, 3, 356, 370)
(63, 356, 370)
Starting IOU-based stitching...
Total unique 3D cells found: 328 via IOU stitching
Stitching iteration 1...
Breaks made this iteration: 37
Stitching iteration 2...
Breaks made this iteration: 11
Stitching iteration 3...
Breaks made this iteration: 2
Stitching iteration 4...
Breaks made this iteration: 1
Stitching iteration 5...
Breaks made this iteration: 0
No more breaks found. Stitching complete.
Total unique 3D cells found: 379 after stitching


RuntimeError: sequence argument must have length equal to input rank

<Labels layer 'IOU stitched mask' at 0x2339107f110>

In [50]:
image = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\cropped\Frame-10.tif"
).astype(np.uint16)
image = image[:,:,:,0:606]
print(image.shape)
crc_image = image[:, 0]
wt_image = image[:, 1]

mask = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\segmented\Frame-10_masks.tif"
)
print(mask.shape)
mask_3d_fixed, IOU_stitched_mask, stitched_df, IOU_stitched_df, _ = stitch_3D(
    wt_image, crc_image, mask=mask, threshold=2.5
)


viewer = napari.Viewer(ndisplay=3)
scale = (2.5, 0.64, 0.64)
viewer.add_image(
    wt_image, scale=scale, name="wt image", visible=False, blending="additive"
)
viewer.add_image(
    crc_image, scale=scale, name="crc image", visible=False, blending="additive"
)
viewer.add_labels(mask_3d_fixed, name="Final stitched mask", scale=scale)
viewer.add_labels(
    IOU_stitched_mask, name="IOU stitched mask", scale=scale, visible=False
)

(92, 3, 528, 606)
(92, 528, 606)
Starting IOU-based stitching...
Total unique 3D cells found: 1205 via IOU stitching
Stitching iteration 1...


,3d_label,labels_2d,z_positions,y_centroids,x_centroids,volumes,raw_wt,raw_crc,volumes_break_score,final_break_score,break_score_up,break_score_down,intensity_break_score,nuclei_intensity,slice_idx
2858,409,7436,31.0,56.189189,142.72973,37.0,137.27027,105.918919,0,0.0,0.0,0.0,0,137.27027,0
2859,409,7856,32.0,56.769231,141.974359,78.0,152.064103,107.564103,1.009533,1.55159,0.587063,1.54715,0.9934,152.064103,1
2860,409,8289,33.0,56.017699,142.371681,113.0,159.938053,107.292035,1.014018,0.887324,0.879655,0.189748,0.994773,159.938053,2
2861,409,8730,34.0,55.37234,142.404255,94.0,167.340426,106.531915,2.122906,1.96183,0.993675,0.774508,0.930007,167.340426,3
2862,409,9182,35.0,54.145161,143.354839,124.0,162.830645,106.758065,1.219953,0.334827,0.276573,0.227497,0.992357,162.830645,4
2863,409,9646,36.0,53.064815,144.018519,108.0,157.231481,106.296296,1.764746,0.078103,0.010445,0.042426,1.043161,157.231481,5
2864,409,10114,37.0,51.902913,144.563107,103.0,147.563107,106.631068,1.940239,2.60235,1.132493,0.63896,1.184336,147.563107,6
2865,409,10583,38.0,51.060976,143.439024,82.0,144.841463,106.378049,3.061273,4.338162,1.152813,0.613306,1.229263,144.841463,7
2866,409,11056,39.0,49.589474,143.336842,95.0,141.894737,105.968421,2.280776,1.89624,0.558145,0.649102,1.280849,141.894737,8
2867,409,11538,40.0,48.847328,143.557252,131.0,146.832061,107.328244,1.199464,0.45802,0.319234,0.044413,1.196159,146.832061,9


Breaks made this iteration: 182
Stitching iteration 2...


,3d_label,labels_2d,z_positions,y_centroids,x_centroids,volumes,raw_wt,raw_crc,volumes_break_score,final_break_score,break_score_up,break_score_down,intensity_break_score,nuclei_intensity,slice_idx
2516,409,7436,31.0,56.189189,142.72973,37.0,137.27027,105.918919,0,0.0,0.0,0.0,0,137.27027,0
2517,409,7856,32.0,56.769231,141.974359,78.0,152.064103,107.564103,0.754109,1.307181,0.587063,1.744928,0.9934,152.064103,1
2518,409,8289,33.0,56.017699,142.371681,113.0,159.938053,107.292035,0.757459,0.66282,0.879655,0.366468,0.994773,159.938053,2
2519,409,8730,34.0,55.37234,142.404255,94.0,167.340426,106.531915,1.585785,1.465463,0.993675,0.296715,0.930007,167.340426,3
2520,409,9182,35.0,54.145161,143.354839,124.0,162.830645,106.758065,0.793704,0.29756,0.276573,0.377788,0.992357,162.830645,4
2521,409,9646,36.0,53.064815,144.018519,108.0,157.231481,106.296296,1.094993,0.052623,0.010445,0.048113,0.998849,157.231481,5
2522,409,10114,37.0,51.902913,144.563107,103.0,147.563107,106.631068,0,0.0,0.0,0.0,0,147.563107,6


Breaks made this iteration: 60
Stitching iteration 3...


...

Breaks made this iteration: 1
Stitching iteration 6...


,3d_label,labels_2d,z_positions,y_centroids,x_centroids,volumes,raw_wt,raw_crc,volumes_break_score,final_break_score,break_score_up,break_score_down,intensity_break_score,nuclei_intensity,slice_idx
2459,409,7436,31.0,56.189189,142.72973,37.0,137.27027,105.918919,0,0.0,0.0,0.0,0,137.27027,0
2460,409,7856,32.0,56.769231,141.974359,78.0,152.064103,107.564103,0.754109,1.307181,0.587063,1.744928,0.9934,152.064103,1
2461,409,8289,33.0,56.017699,142.371681,113.0,159.938053,107.292035,0.757459,0.66282,0.879655,0.366468,0.994773,159.938053,2
2462,409,8730,34.0,55.37234,142.404255,94.0,167.340426,106.531915,1.585785,1.465463,0.993675,0.296715,0.930007,167.340426,3
2463,409,9182,35.0,54.145161,143.354839,124.0,162.830645,106.758065,0.793704,0.29756,0.276573,0.377788,0.992357,162.830645,4
2464,409,9646,36.0,53.064815,144.018519,108.0,157.231481,106.296296,1.094993,0.052623,0.010445,0.048113,0.998849,157.231481,5
2465,409,10114,37.0,51.902913,144.563107,103.0,147.563107,106.631068,0,0.0,0.0,0.0,0,147.563107,6


Breaks made this iteration: 0
No more breaks found. Stitching complete.
Total unique 3D cells found: 1479 after stitching


RuntimeError: sequence argument must have length equal to input rank

<Labels layer 'IOU stitched mask' at 0x23439ed97d0>

In [49]:
import numpy as np
from skimage.measure import regionprops


def centroid_accuracy(perfect_mask, mask_3d_fixed):
    perfect_mask = np.asarray(perfect_mask)
    pred_mask = np.asarray(mask_3d_fixed)
    if perfect_mask.shape != pred_mask.shape:
        raise ValueError("Masks must share shape")

    gt_props = regionprops(perfect_mask)
    hit_pred_labels = set()
    predicted_labels = np.array([])
    tp = fn = 0

    for prop in gt_props:
        centroid = np.rint(prop.centroid).astype(int)
        centroid = tuple(np.clip(centroid, 0, np.array(perfect_mask.shape) - 1))
        pred_label = pred_mask[centroid]
        predicted_labels = np.append(predicted_labels, pred_label)
        if pred_label > 0:
            tp += 1
            hit_pred_labels.add(int(pred_label))
        else:
            fn += 1

    pred_labels = set(np.unique(pred_mask))
    pred_labels.discard(0)
    fp = len(pred_labels - hit_pred_labels)
    fp_list = list(pred_labels - hit_pred_labels)
    merged_cells = len(predicted_labels) - len(np.unique(predicted_labels))

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0

    gt_binary = perfect_mask > 0
    pred_binary = pred_mask > 0
    intersection = np.count_nonzero(gt_binary & pred_binary)
    dice = (
        (2 * intersection) / (gt_binary.sum() + pred_binary.sum())
        if (gt_binary.sum() + pred_binary.sum())
        else 0.0
    )

    return {
        "true_positives": tp,
        "false_negatives": fn,
        "false_positives": fp,
        "merged_cells": merged_cells,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "voxel_dice": dice,
        "false_positive_labels": fp_list,
    }

# print(centroid_accuracy(perfect_mask, mask_3d_fixed))
# print(f"actual amount of cells: {len(np.unique(perfect_mask)) - 1}, predicted amount of cells: {len(np.unique(mask_3d_fixed)) - 1}")

In [114]:
# Method 2: Access the actual list value
cell = 79
intensity = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell][
    "nuclei_intensity"
].values[0])
intensity_break = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["intensity_break_score"].values[0])
volume = df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["volumes"].values[0]
volume_break = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["volumes_break_score"].values[0])
break_score_up = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["break_score_up"].values[0])
break_score_down = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["break_score_down"].values[0])
final_score = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["final_break_score"].values[0])

# create df of this and display
df_cell_detail = pd.DataFrame(
    {
        "slice": range(len(volume)),
        "volume": volume,
        "nuclei_intensity": intensity,
        "intensity_break_score": intensity_break,
        "volume_break_score": volume_break,
        "break_score_up": break_score_up,
        "break_score_down": break_score_down,
        "final_break_score": final_score,
    }
)



display(df_cell_detail)

,slice,volume,nuclei_intensity,intensity_break_score,volume_break_score,break_score_up,break_score_down,final_break_score
0,0,66.0,21.500000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1,138.0,36.050725,1.653241,0.693132,0.138247,0.306124,0.350792
2,2,174.0,59.528736,1.016685,0.911613,0.427690,0.048959,0.396393
3,3,189.0,84.000000,0.843128,0.974217,0.002691,0.002606,0.002210
4,4,200.0,97.690000,0.879639,0.897750,0.087896,0.084598,0.069411
5,5,190.0,99.936842,0.854310,1.008310,0.373727,0.354597,0.321932
6,6,182.0,87.340659,0.894629,1.086825,0.066677,0.057741,0.064830
7,7,180.0,68.288889,0.000000,0.000000,0.000000,0.000000,0.000000


In [115]:
import napari

viewer = napari.Viewer(ndisplay=3)

# Create a shared color dictionary for specific labels
import numpy as np

wt_image = crc_image = tifffile.imread(os.path.join(output_directory, "realistic_organoid.tif"))
perfect_image = tifffile.imread(os.path.join(output_directory, "perfect_organoid.tif"))

# Generate colors for labels 0-max_label
max_label = max(mask_3d.max(), mask_3d_fixed.max())
np.random.seed(42)  # For reproducible colors
color_dict = {}
for i in range(max_label + 1):
    # Generate bright colors with full alpha
    color = np.random.rand(3)  # RGB only
    color = (
        color / color.max()
    )  # Normalize so at least one channel is at max brightness
    color_dict[i] = np.append(color, 0.8)  # Add full alpha

color_dict[0] = np.array([0, 0, 0, 0])  # Background transparent

# Apply to both layers
viewer.add_labels(
    mask_3d, name="3D_stitched_mask", scale=(2.5, 0.64, 0.64), colormap=color_dict
)
viewer.add_labels(
    mask_3d_fixed,
    name="3D_stitched_mask_fixed",
    scale=(2.5, 0.64, 0.64),
    colormap=color_dict,
)
viewer.add_labels(
    mask_3d_cuts,
    name="3D_stitched_mask_cuts",
    scale=(2.5, 0.64, 0.64),
    colormap=color_dict,
)

viewer.add_labels(
    unique_mask, name="original_mask", scale=(2.5, 0.64, 0.64), visible=False
)
viewer.add_image(
    crc_image,
    name="crc_image",
    scale=(2.5, 0.64, 0.64),
    blending="additive",
    visible=False,
)
viewer.add_image(
    wt_image,
    name="wt_image",
    scale=(2.5, 0.64, 0.64),
    blending="additive",
    visible=False,
)
viewer.add_image(
    perfect_image,
    name="perfect_image",
    scale=(2.5, 0.64, 0.64),
    blending="additive",
    visible=False,
)

viewer.add_labels(
    perfect_mask, name="perfect_mask", scale=(2.5, 0.64, 0.64), visible=False
)

c:\Users\6331823\AppData\Local\Programs\Python\Python311\Lib\site-packages\napari\utils\colormaps\colormap.py:447: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
c:\Users\6331823\AppData\Local\Programs\Python\Python311\Lib\site-packages\napari\utils\colormaps\colormap.py:447: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
c:\Users\6331823\AppData\Local\Programs\Python\Python311\Lib\site-packages\napari\utils\colormaps\colormap.py:447: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(


RuntimeError: sequence argument must have length equal to input rank

RuntimeError: sequence argument must have length equal to input rank

<Labels layer 'perfect_mask' at 0x16696f9a810>

In [86]:
from skimage.draw import ellipsoid
import napari
import os
from scipy.ndimage import zoom, gaussian_filter

def make_cell(radius_Z, radius_Y, radius_X, intensity=255):
    """Create a binary ellipsoidal cell"""
    cell = ellipsoid(radius_Z, radius_Y, radius_X)
    cell = cell.astype(np.uint16) * intensity
    return cell


def paste_cell(volume, mask, cell, position, label):
    cz, cy, cx = position.astype(int)
    dz, dy, dx = cell.shape
    hz, hy, hx = dz // 2, dy // 2, dx // 2

    z0, z1 = max(0, cz - hz), min(volume.shape[0], cz + hz + 1)
    y0, y1 = max(0, cy - hy), min(volume.shape[1], cy + hy + 1)
    x0, x1 = max(0, cx - hx), min(volume.shape[2], cx + hx + 1)

    cell_z0 = hz - (cz - z0)
    cell_y0 = hy - (cy - y0)
    cell_x0 = hx - (cx - x0)

    subcell = cell[
        cell_z0 : cell_z0 + (z1 - z0),
        cell_y0 : cell_y0 + (y1 - y0),
        cell_x0 : cell_x0 + (x1 - x0),
    ]

    if subcell.size == 0:
        return False

    volume[z0:z1, y0:y1, x0:x1] = np.maximum(volume[z0:z1, y0:y1, x0:x1], subcell)
    mask[z0:z1, y0:y1, x0:x1][subcell > 0] = label
    return True

from scipy.ndimage import affine_transform


def _rotation_matrix(from_vec, to_vec):
    a = from_vec / (np.linalg.norm(from_vec) + 1e-8)
    b = to_vec / (np.linalg.norm(to_vec) + 1e-8)
    v = np.cross(a, b)
    c = np.dot(a, b)
    if np.isclose(c, 1.0):
        return np.eye(3)
    if np.isclose(c, -1.0):
        # 180° flip around any perpendicular axis
        perp = np.array([1.0, 0.0, 0.0])
        if np.allclose(a, perp):
            perp = np.array([0.0, 1.0, 0.0])
        v = perp - a * np.dot(perp, a)
        v /= np.linalg.norm(v)
        vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        return np.eye(3) + 2 * vx @ vx
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx * ((1 - c) / (np.linalg.norm(v) ** 2 + 1e-8))


def orient_cell(cell, direction_vec):
    base = np.array([1.0, 0.0, 0.0])  # +z axis in (z, y, x) ordering
    pad = int(max(cell.shape) * 0.75)
    padded = np.pad(
        cell,
        ((pad, pad), (pad, pad), (pad, pad)),
        mode="constant",
        constant_values=0,
    )
    R = _rotation_matrix(base, direction_vec)
    center = (np.array(padded.shape) - 1) / 2
    matrix = R.T
    offset = center - matrix @ center
    rotated = affine_transform(
        padded,
        matrix=matrix,
        offset=offset,
        order=1,
        mode="constant",
        cval=0,
    )
    return rotated.astype(cell.dtype)


def can_place(mask, cell, position):
    cz, cy, cx = position.astype(int)
    dz, dy, dx = cell.shape
    hz, hy, hx = dz // 2, dy // 2, dx // 2

    z0, z1 = max(0, cz - hz), min(mask.shape[0], cz + hz + 1)
    y0, y1 = max(0, cy - hy), min(mask.shape[1], cy + hy + 1)
    x0, x1 = max(0, cx - hx), min(mask.shape[2], cx + hx + 1)

    if z0 >= z1 or y0 >= y1 or x0 >= x1:
        return False

    cell_z0 = hz - (cz - z0)
    cell_y0 = hy - (cy - y0)
    cell_x0 = hx - (cx - x0)

    subcell = cell[
        cell_z0 : cell_z0 + (z1 - z0),
        cell_y0 : cell_y0 + (y1 - y0),
        cell_x0 : cell_x0 + (x1 - x0),
    ]

    return np.all(mask[z0:z1, y0:y1, x0:x1][subcell > 0] == 0)

number_of_cells = 250
max_attempts = 30
radius_X = radius_Y = 12
radius_Z = 5
randomness = 0.2

vol_shape = (300,300,300)
volume = np.zeros(vol_shape, dtype=np.uint16)
mask = np.zeros_like(volume)

organoid_size = 250

viewer = napari.Viewer(ndisplay=3)

for cell_idx in range(number_of_cells):
    # create a binomial distrubution around the radii with the arms reaching the randomness value
    rand_radius_X = int(radius_X + np.random.uniform(-randomness, randomness) * radius_X)
    rand_radius_Y = int(radius_Y + np.random.uniform(-randomness, randomness) * radius_Y)
    rand_radius_Z = int(radius_Z + np.random.uniform(-randomness, randomness) * radius_Z)
    cell = make_cell(rand_radius_Z, rand_radius_Y, rand_radius_X, intensity=255)

    placed = False
    for attempt in range(max_attempts):
        # get a random position at distance of organoid_size/2 from center
        u = np.random.uniform(-1, 1)  # cos(theta)
        phi = np.random.uniform(0, 2 * np.pi)
        theta = np.arccos(u)
        r = organoid_size / 2 # controls how far from center the cells are placed
        pos_z = int(center[0] + r * u)
        sin_theta = np.sqrt(1 - u * u)
        pos_y = int(center[1] + r * sin_theta * np.cos(phi))
        pos_x = int(center[2] + r * sin_theta * np.sin(phi))
        position = np.array([pos_z, pos_y, pos_x])

        # place the cell in the volume at the random position
        direction = center - position  # point toward organoid center
        oriented_cell = orient_cell(cell, direction)

        if not can_place(mask, oriented_cell, position):
            continue

        paste_success = paste_cell(volume, mask, oriented_cell, position, label=cell_idx + 1)
        if paste_success:
            placed = True
            print(
                f"Cell {cell_idx+1}: position=({pos_z}, {pos_y}, {pos_x}), radii=({rand_radius_Z}, {rand_radius_Y}, {rand_radius_X})"
            )
            break

    if not placed:
        print(f"Cell {cell_idx}: skipped after {max_attempts} attempts")

rescale = ((vol_shape[0] / (2.5 / 0.64) / vol_shape[0]), 1.0, 1.0)
volume = zoom(volume, zoom=rescale, order=1)
mask = zoom(mask, zoom=rescale, order=0)
output_directory = r"C:\Users\6331823\Local SSD\Fake_data"
tifffile.imwrite(os.path.join(output_directory, "perfect_organoid.tif"), volume)
tifffile.imwrite(os.path.join(output_directory, "perfect_mask.tif"), mask)

unique_labels = np.unique(mask)
unique_labels = unique_labels[unique_labels > 0]

rng = np.random.default_rng()

brightness_volume = np.zeros_like(volume, dtype=np.float32)

for lbl in unique_labels:
    cell_voxels = mask == lbl
    cell_mean = np.clip(rng.normal(200, 15), 150, 250)
    cell_noise = rng.normal(loc=cell_mean, scale=5, size=cell_voxels.sum())
    lo = max(150, cell_mean - 10)
    hi = min(250, cell_mean + 10)
    brightness_volume[cell_voxels] = np.clip(cell_noise, lo, hi)

brightness_volume = brightness_volume.astype(np.uint16)
tifffile.imwrite(os.path.join(output_directory, "brightness_adjusted_organoid.tif"), brightness_volume)

gaussian_volume = gaussian_filter(
    brightness_volume.astype(np.float32), sigma=(1.6, 0.6, 0.6)
)
tifffile.imwrite(
    os.path.join(output_directory, "gaussian_organoid.tif"),
    gaussian_volume.astype(np.uint16),
)

z_profile = np.linspace(0.9, 0.3, volume.shape[0], dtype=np.float32)[:, None, None]
realistic_volume = gaussian_volume.astype(np.float32) * z_profile
realistic_volume = np.clip(realistic_volume, 0, 255).astype(np.uint16)
tifffile.imwrite(
    os.path.join(output_directory, "realistic_organoid.tif"),
    realistic_volume.astype(np.uint16),
)

scale = (2.5, 0.64, 0.64)
viewer.add_image(volume, name="synthetic_volume", scale=scale)
viewer.add_labels(mask, name="synthetic_mask", scale=scale)
viewer.add_image(brightness_volume, name="brightness_volume", scale=scale)
viewer.add_image(gaussian_volume, name="gaussian_volume", scale=scale)
viewer.add_image(realistic_volume, name="realistic_volume", scale=scale)

Cell 1: position=(47, 201, 198), radii=(4, 10, 10)
Cell 2: position=(150, 104, 33), radii=(4, 10, 11)
Cell 3: position=(45, 198, 101), radii=(5, 13, 9)
Cell 4: position=(226, 148, 51), radii=(4, 9, 14)
Cell 5: position=(146, 57, 65), radii=(4, 10, 10)
Cell 6: position=(34, 149, 196), radii=(5, 11, 11)
Cell 7: position=(158, 247, 227), radii=(5, 13, 13)
Cell 8: position=(92, 70, 227), radii=(4, 12, 11)
Cell 9: position=(106, 235, 230), radii=(4, 11, 9)
Cell 10: position=(222, 48, 150), radii=(5, 12, 13)
Cell 11: position=(211, 50, 195), radii=(5, 12, 10)
Cell 12: position=(186, 42, 98), radii=(4, 10, 10)
Cell 13: position=(84, 235, 213), radii=(5, 14, 11)
Cell 14: position=(71, 128, 245), radii=(4, 10, 10)
Cell 15: position=(156, 44, 216), radii=(4, 13, 10)
Cell 16: position=(267, 178, 117), radii=(4, 10, 14)
Cell 17: position=(192, 256, 99), radii=(4, 10, 13)
Cell 18: position=(217, 190, 247), radii=(4, 12, 12)
Cell 19: position=(177, 120, 268), radii=(5, 11, 11)
Cell 20: position=(158

<Image layer 'realistic_volume' at 0x161df084e10>

In [103]:
from scipy.ndimage import gaussian_filter, map_coordinates

# axial elongation / blur
brightness_volume = tifffile.imread(os.path.join(output_directory, "brightness_adjusted_organoid.tif"))


scale = (2.5, 0.64, 0.64)
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(brightness_volume, name="synthetic_volume", scale=scale)
viewer.add_labels(mask, name="synthetic_mask", scale=scale)
viewer.add_image(brightness_volume, name="brightness_volume", scale=scale)

<Image layer 'brightness_volume' at 0x1666b052350>

In [102]:
gaussian_volume = gaussian_filter(
    brightness_volume.astype(np.float32), sigma=(3, 0.6, 0.6)
)
z_profile = np.linspace(0.9, 0.3, volume.shape[0], dtype=np.float32)[:, None, None]
realistic_volume = gaussian_volume.astype(np.float32) * z_profile
realistic_volume = np.clip(realistic_volume, 0, 255).astype(np.uint16)
viewer = napari.Viewer(ndisplay=3)
scale = (2.5, 0.64, 0.64)
viewer.add_image(volume, name="synthetic_volume", scale=scale)
viewer.add_labels(mask, name="synthetic_mask", scale=scale)
viewer.add_image(brightness_volume, name="brightness_volume", scale=scale)
viewer.add_image(gaussian_volume, name="gaussian_volume", scale=scale)
viewer.add_image(realistic_volume, name="realistic_volume", scale=scale)

AttributeError: 'list' object has no attribute 'shape'

Cleaner version of organoid stitcher

In [31]:
import tifffile
import pandas as pd
from skimage.measure import regionprops, regionprops_table
import napari
import numpy as np
import os
from dataclasses import dataclass
import scipy.spatial


@dataclass
class cell_3d:
    label: int
    centroid: tuple
    coords_set: set
    volume: int
    labels_2d: list | None = None
    centroids_2d: list | None = None
    coords_set_2d: list | None = None
    volumes_2d: list | None = None
    # per-slice per-channel intensities, dict of lists keyed by channel name
    intensities_2d: dict | None = None
    # aggregated per-channel intensities (mean across slices), keys like "intensity_wt"
    intensities: dict | None = None
    break_scores: list | None = None
    parents: list | None = None
    touching_cells: list | None = None


@dataclass
class cell_2d:
    label: int
    centroid: tuple
    coords_set: set
    volume: int

    def __post_init__(self):
        self._intensities = {}


def get_cell_properties_2d(image):
    # Get the properties of every mask in the image
    props = regionprops(image)

    organoid = []

    # For every mask found, get the label, centeroid, boundingbox, and volume
    for prop in props:
        # Pre-compute coordinate set for fast IOU calculations later
        coords_2d = prop.coords[:, 1:]
        coords_set = set(map(tuple, coords_2d))

        cell = cell_2d(
            label=prop.label,
            centroid=prop.centroid,  # (z, y, x)
            coords_set=coords_set,
            volume=prop.area,
        )
        organoid.append(cell)

    return organoid


def properties_channel(organoid, mask, image, channel_type=None):
    props_channel = regionprops_table(
        mask, intensity_image=image, properties=["label", "mean_intensity"]
    )
    intensity_dict = dict(zip(props_channel["label"], props_channel["mean_intensity"]))

    for cell in organoid:
        attr_name = f"intensity_{channel_type}" if channel_type else "intensity"
        setattr(cell, attr_name, intensity_dict.get(cell.label, 0))

    return organoid


def make_unique_mask(mask):
    """Makes a fresh mask from a already stitched mask"""
    unique_mask = []
    number = 1
    for z in range(mask.shape[0]):
        # go over every unique number in the mask and create a new slice with a new unique number 1 higher
        slice = mask[z]
        unique_slice = mask[z].copy()
        for val in range(1, slice.max() + 1):
            unique_slice[slice == val] = number
            number += 1
        unique_mask.append(unique_slice)
    unique_mask = np.stack(unique_mask, axis=0)

    return unique_mask

def get_2d_mask_properties(mask, image1, image_type_1=None, image2=None, image_type_2=None):
    unique_mask = make_unique_mask(mask)
    organoid = get_cell_properties_2d(unique_mask)

    organoid = properties_channel(organoid, unique_mask, image1, channel_type=image_type_1)

    if image2 is not None:
        organoid = properties_channel(
            organoid, unique_mask, image2, channel_type=image_type_2
        )

    return organoid


def calculate_iou(coords_sets, idx1, idx2):
    """
    Calculate overlap metric for cell matching.
    Returns 1 if the smaller cell is completely contained in the larger cell.
    Otherwise returns intersection over union (standard IOU).
    """
    set1 = coords_sets[idx1]
    set2 = coords_sets[idx2]

    # Check if either set is empty
    len1, len2 = len(set1), len(set2)
    if len2 == 0:
        return 0

    # Calculate intersection (overlapping pixels)
    intersection = len(set1 & set2)

    if intersection == 0:
        return 0  # No overlap = different cells

    # If the smaller cell is completely contained in the larger cell, return 1
    min_len = min(len1, len2)
    if intersection == min_len:
        return 1.0  # One cell completely covers the other

    # Otherwise, use standard IOU
    union = len1 + len2 - intersection
    return intersection / union


def IOU_stitching(organoid_2d, distance_threshold=10, iou_threshold=0.3):
    print("Starting IOU based 3D stitching...")
    n = len(organoid_2d)
    z_array = np.array([c.centroid[0] for c in organoid_2d])
    y_array = np.array([c.centroid[1] for c in organoid_2d])
    x_array = np.array([c.centroid[2] for c in organoid_2d])
    coords_sets = {i: c.coords_set for i, c in enumerate(organoid_2d)}

    # group by integer z to increase speed
    z_groups = {}
    for idx, z_val in enumerate(z_array):
        z_int = int(round(z_val))
        z_groups.setdefault(z_int, []).append(idx)

    labels_3d = np.full(n, -1, dtype=np.int32)

    def stitch_forward(idx, current_label):
        labels_3d[idx] = current_label
        next_z = int(round(z_array[idx])) + 1
        if next_z not in z_groups:
            return
        # get all cells that havent been stitched yet
        candidates = [c for c in z_groups[next_z] if labels_3d[c] == -1] 
        if not candidates:
            return
        cand_arr = np.array(candidates)
        dists = np.hypot(
            x_array[cand_arr] - x_array[idx], y_array[cand_arr] - y_array[idx]
        )
        closest_cell = np.argmin(dists)
        if dists[closest_cell] > distance_threshold:
            return
        cand_idx = cand_arr[closest_cell]
        if calculate_iou(coords_sets, idx, cand_idx) > iou_threshold:
            stitch_forward(cand_idx, current_label)

    current_label = 0
    for i in range(n):
        if labels_3d[i] == -1:
            current_label += 1
            stitch_forward(i, current_label)

    # build organoid_3d list
    organoid_3d = []
    intensity_attrs = [a for a in organoid_2d[0].__dict__ if a.startswith("intensity")]
    for cell in np.unique(labels_3d):
        if cell <= 0:
            continue
        slice_idxs = np.where(labels_3d == cell)[0]
        labels_2d = [organoid_2d[i].label for i in slice_idxs]
        centroids_2d = [organoid_2d[i].centroid for i in slice_idxs]
        coords_set_2d = [organoid_2d[i].coords_set for i in slice_idxs]
        volumes_2d = [organoid_2d[i].volume for i in slice_idxs]
        # collect per-channel per-slice intensities
        intensities_2d = {
            attr: [getattr(organoid_2d[i], attr, None) for i in slice_idxs]
            for attr in intensity_attrs
        }

        coords_3d = set()
        for z_idx, (z, _, _) in enumerate(centroids_2d):
            z_int = int(round(z))
            for y, x in coords_set_2d[z_idx]:
                coords_3d.add((z_int, int(y), int(x)))

        cell3d = cell_3d(
            label=cell,
            centroid=np.mean(centroids_2d, axis=0),
            coords_set=coords_3d,
            volume=sum(volumes_2d),
            labels_2d=labels_2d,
            centroids_2d=centroids_2d,
            coords_set_2d=coords_set_2d,
            volumes_2d=volumes_2d,
            intensities_2d=intensities_2d,
        )
        organoid_3d.append(cell3d)

    print(f"Completed IOU based stitching: {len(organoid_3d)} cells formed.")

    return organoid_3d

def mask_from_organoid(organoid, shape):
    mask = np.zeros(shape, dtype=np.uint16)
    for cell in organoid:
        for z_idx, (z, y, x) in enumerate(cell.centroids_2d):
            z_int = int(round(z))
            if 0 <= z_int < shape[0]:
                for yy, xx in cell.coords_set_2d[z_idx]:
                    if 0 <= yy < shape[1] and 0 <= xx < shape[2]:
                        mask[z_int, yy, xx] = cell.label
    return mask


def break_score(values, i):
    # Calculate intensity-based anomalies by getting a score that resembles high-low-high intensity value patterns
    # Calculate ratio between slice i and below i, and i and above i
    values_bottom = max(values[:i])
    values_top = max(values[i + 1 :])
    ratio_1 = values_bottom / values[i]
    ratio_2 = values[i] / values_top

    # divide this again, now high low high gets higher score then other patterns
    return ratio_1 / ratio_2

def get_centroid_positions(centroid):
    z_positions = [c[0] for c in centroid]
    y_positions = [c[1] for c in centroid]
    x_positions = [c[2] for c in centroid]
    return z_positions, y_positions, x_positions


def calculate_line_3d(z_positions, y_positions, x_positions, points=3):
    # Fit line to specified points (int, tuple, or slice)
    if isinstance(points, tuple):
        points = slice(*points)
    elif isinstance(points, int):
        points = slice(None, points)

    z_fit = np.array(z_positions[points])
    x_fit = np.array(x_positions[points])
    y_fit = np.array(y_positions[points])

    # Calculate line parameters
    A = np.vstack([z_fit, np.ones(len(z_fit))]).T
    m_x, c_x = np.linalg.lstsq(A, x_fit, rcond=None)[0]
    m_y, c_y = np.linalg.lstsq(A, y_fit, rcond=None)[0]

    # Create line function: given z, returns (x, y) in that order
    line_function = lambda z: (m_x * z + c_x, m_y * z + c_y)

    return line_function


def calculate_point_line_distances(
    z_positions, y_positions, x_positions, line_function
):
    distances = []
    for z, x, y in zip(z_positions, x_positions, y_positions):
        line_x, line_y = line_function(z)
        dist = np.sqrt((x - line_x) ** 2 + (y - line_y) ** 2)
        distances.append(dist)
    return distances

def get_slope_changes(distances):
    # Calculate slope changes in the distance array
    slopes = np.diff(distances)
    slope_changes = np.diff(slopes)

    # Pad to match original length
    slope_changes = np.concatenate([[0], slope_changes, [0]])
    return slope_changes

def get_volume_break_scores(cell):
    num_slices = len(cell.volumes_2d)
    volumes_break_score = [0] * num_slices

    for i in range(1, num_slices - 1):
        volumes_break_score[i] = break_score(cell.volumes_2d, i)

    return volumes_break_score

def get_intensity_break_scores(cell):
    num_slices = len(cell.volumes_2d)
    intensity_break_score = [0] * num_slices

    if cell.intensities_2d:
        per_channel = list(cell.intensities_2d.values())
        max_intensity = [max(vals) for vals in zip(*per_channel)]
    else:
        max_intensity = [0] * num_slices

    for i in range(1, num_slices - 1):
        intensity_break_score[i] = break_score(max_intensity, i)

    return intensity_break_score

def get_line_break_scores(z_positions, y_positions, x_positions):
    # Calculate line-based break scores (from bottom_up)
    line = calculate_line_3d(z_positions, y_positions, x_positions, points=3)
    distances = calculate_point_line_distances(z_positions, y_positions, x_positions, line)
    break_score = abs(get_slope_changes(distances))
    return break_score

def get_IOU_break_scores(cell):
    num_slices = len(cell.volumes_2d)
    iou_break_score = [0] * num_slices

    for i in range(1, num_slices - 1):
        iou_break_score[i] = 1 - calculate_iou(
            cell.coords_set_2d, i - 1, i
        )

    return iou_break_score

def get_break_scores(organoid):
    for cell in organoid:
        num_slices = len(cell.volumes_2d)
        if num_slices < 5:
            cell.break_scores = [0] * num_slices
            continue

        # Calculate volume-based break scores
        volumes_break_score = get_volume_break_scores(cell)
        # Calculate intensity-based break scores
        intensity_break_score = get_intensity_break_scores(cell)

        # Calculate line-based break scores (from bottom_up)
        z_positions, y_positions, x_positions = get_centroid_positions(cell.centroids_2d)
        break_score_up = get_line_break_scores(z_positions, y_positions, x_positions)
        # pad 1 at start and remove last to match and better represent breaks between slices
        break_score_up = np.concatenate([[0], break_score_up[:-1]])
        # Calculate line-based break scores (from top_down)
        break_score_down = get_line_break_scores(
            z_positions[::-1], y_positions[::-1], x_positions[::-1]
        )[::-1]
        IOU_break_score = get_IOU_break_scores(cell)

        # Calculate final break scores
        final_scores = []
        for v_score, i_score, up_score, down_score, IOU_score in zip(
            volumes_break_score,
            intensity_break_score,
            break_score_up,
            break_score_down,
            IOU_break_score,
        ):
            final_score = np.max([up_score, down_score]) * v_score * (i_score**2) * (IOU_score*2)
            # if cell.label == 83:
            #     print(f"v:{v_score:.2f}, i:{i_score:.2f}, up:{up_score:.2f}, down:{down_score:.2f}, IOU:{IOU_score:.2f}, final:{final_score:.2f}")
            final_scores.append(final_score)

        cell.break_scores = final_scores

    return organoid

def split_cell_at_index(cell, break_position, new_label):
    # Build parent lineage: existing parents + current label
    parent_lineage = (cell.parents or []) + [cell.label]

    # Build 3D coords_set for bottom cell
    coords_3d_bottom = set()
    for z_idx in range(break_position):
        z_int = int(round(cell.centroids_2d[z_idx][0]))
        for y, x in cell.coords_set_2d[z_idx]:
            coords_3d_bottom.add((z_int, int(y), int(x)))

    bottom_cell = cell_3d(
        label=cell.label,
        centroid=np.mean(cell.centroids_2d[:break_position], axis=0),
        coords_set=coords_3d_bottom,
        volume=sum(cell.volumes_2d[:break_position]),
        labels_2d=cell.labels_2d[:break_position],
        centroids_2d=cell.centroids_2d[:break_position],
        coords_set_2d=cell.coords_set_2d[:break_position],
        volumes_2d=cell.volumes_2d[:break_position],
        intensities_2d={
            k: v[:break_position] for k, v in (cell.intensities_2d or {}).items()
        },
        parents=parent_lineage,
    )

    # Build 3D coords_set for top cell
    max_z = len(cell.volumes_2d)
    coords_3d_top = set()
    for z_idx in range(break_position, max_z):
        z_int = int(round(cell.centroids_2d[z_idx][0]))
        for y, x in cell.coords_set_2d[z_idx]:
            coords_3d_top.add((z_int, int(y), int(x)))

    top_cell = cell_3d(
        label=new_label,
        centroid=np.mean(cell.centroids_2d[break_position:max_z], axis=0),
        coords_set=coords_3d_top,
        volume=sum(cell.volumes_2d[break_position:max_z]),
        labels_2d=cell.labels_2d[break_position:max_z],
        centroids_2d=cell.centroids_2d[break_position:max_z],
        coords_set_2d=cell.coords_set_2d[break_position:max_z],
        volumes_2d=cell.volumes_2d[break_position:max_z],
        intensities_2d={
            k: v[break_position:max_z] for k, v in (cell.intensities_2d or {}).items()
        },
        parents=parent_lineage,
    )

    return bottom_cell, top_cell

def break_stitching(organoid, breaking_threshold = 2.5):
    new_organoid = []
    breaks_made = 0
    new_label = max(cell.label for cell in organoid) + 1
    for cell in organoid:
        if len(cell.volumes_2d) < 7:
            new_organoid.append(cell)
            continue
        # if cell.label == 103:
        #     print(cell.break_scores)
        # Find indices where break score exceeds threshold
        break_scores = [score for score in cell.break_scores]
        break_scores[:3] = [0, 0, 0]  # First 3 slices get score 0
        break_scores[-3:] = [0, 0, 0]  # Last 3 slices get score 0
        if max(break_scores) < breaking_threshold:
            new_organoid.append(cell)
            continue

        # break indice is the highest scoring indices above threshold
        max_break_value = max(break_scores)
        break_position = break_scores.index(max_break_value)

        # Create new cells by breaking at the identified indices
        bottom_cell, top_cell = split_cell_at_index(cell, break_position, new_label=new_label)
        new_organoid.append(bottom_cell)
        new_organoid.append(top_cell)
        new_label += 1
        breaks_made += 1
    return new_organoid, breaks_made


def find_touching_pairs(organoid):
    neighbor_offsets = [(-1, 0, 0), (1, 0, 0)]  # Only z-direction neighbors

    # Build spatial index using KDTree for fast nearest neighbor search
    centroids = np.array([cell.centroid for cell in organoid])
    tree = scipy.spatial.cKDTree(centroids)

    for i, cell in enumerate(organoid):
        if len(cell.volumes_2d) <= 1:
            cell.touching_cells = []
            continue

        # Query k=10 nearest neighbors (increase if needed, but 5-10 is usually enough)
        distances, indices = tree.query(cell.centroid, k=min(5, len(organoid)))

        # Skip self (first result is always self)
        candidate_indices = indices[1:]

        # Check only nearby candidates for actual touching
        touching_cells = []
        current_boundary = cell.coords_set

        for idx in candidate_indices:
            other_cell = organoid[idx]
            other_coords = other_cell.coords_set

            # Fast boundary check using set operations
            # Create neighbor coordinates for current cell
            neighbor_coords = {
                (z + dz, y, x)
                for z, y, x in current_boundary
                for dz, _, _ in neighbor_offsets
            }

            # Check if any neighbors overlap with other cell
            if neighbor_coords & other_coords:  # Set intersection
                touching_cells.append(other_cell.label)

                # Early exit if we found enough touching cells
                if len(touching_cells) >= 5:
                    break

        cell.touching_cells = touching_cells

    return organoid


def find_touching_cells(organoid):
    # return a list of unique pairs of touching cells
    touching_pairs = set()
    for cell in organoid:
        for touching_label in cell.touching_cells:
            pair = tuple(sorted((cell.label, touching_label)))
            touching_pairs.add(pair)

    return list(touching_pairs)


def calculate_split_coords(cell, index, pred_point_1, pred_point_2):
    # Create splitting line perpendicular to the line between the two predicted points
    # Direction vector between predicted points
    direction = pred_point_2 - pred_point_1
    # Perpendicular vector (rotate 90 degrees)
    normal = direction / (np.linalg.norm(direction) + 1e-8)
    # Midpoint between predicted points
    midpoint = (pred_point_1 + pred_point_2) / 2

    part_1_coords = []
    part_2_coords = []

    for coord in cell.coords_set_2d[index]:
        y, x = coord
        point = np.array([y, x])
        # Vector from midpoint to this point
        to_point = point - midpoint
        # Dot product determines which side of the line
        side = np.dot(to_point, normal)

        if side <= 0:
            part_1_coords.append(coord)
        else:
            part_2_coords.append(coord)

    # Calculate new centroids
    part_1_coords = np.array(part_1_coords)
    part_2_coords = np.array(part_2_coords)

    return part_1_coords, part_2_coords


def are_split_cells_valid(cell, index, part_1_coords, part_2_coords):
    min_size = max(
        3, len(cell.coords_set_2d[index]) * 0.1
    )  # At least 10% of pixels or 3 pixels

    if len(part_1_coords) < min_size or len(part_2_coords) < min_size:
        # print(
        #     f"  Warning: Split too unbalanced ({len(part_1_coords)} vs {len(part_2_coords)} pixels)"
        # )
        return False
    return True


def calculate_new_distances(part_1_coords, part_2_coords, pred_point_1, pred_point_2):
    centroid_1 = np.mean(part_1_coords, axis=0)  # (y, x)
    centroid_2 = np.mean(part_2_coords, axis=0)  # (y, x)

    # Calculate distances from new centroids to predicted points
    dist_c1_to_pred1 = np.sqrt(
        (centroid_1[0] - pred_point_1[0]) ** 2 + (centroid_1[1] - pred_point_1[1]) ** 2
    )
    dist_c2_to_pred2 = np.sqrt(
        (centroid_2[0] - pred_point_2[0]) ** 2 + (centroid_2[1] - pred_point_2[1]) ** 2
    )
    # Check if splitting improves the fit
    avg_new_distance = (dist_c1_to_pred1 + dist_c2_to_pred2) / 2

    return avg_new_distance


def check_split_cells(cell, index, pred_point_1, pred_point_2):
    # centroids_2d stores (z, y, x), so extract y, x
    original_centroid = cell.centroids_2d[index]
    original_y = original_centroid[1]  # y is at index 1
    original_x = original_centroid[2]  # x is at index 2

    original_distance = np.sqrt(
        (original_y - pred_point_1[0]) ** 2 + (original_x - pred_point_1[1]) ** 2
    )

    part_1_coords, part_2_coords = calculate_split_coords(
        cell, index, pred_point_1, pred_point_2
    )


    if not are_split_cells_valid(cell, index, part_1_coords, part_2_coords):
        # print(f"    ✗ Split cells not valid")
        return None

    avg_new_distance = calculate_new_distances(
        part_1_coords, part_2_coords, pred_point_1, pred_point_2
    )

    if avg_new_distance < original_distance:
        # print(
        #     f"    ✓ Splitting improves fit! cell {cell.label} (reduction: {original_distance - avg_new_distance:.2f})"
        # )
        return (part_1_coords, part_2_coords)
    else:
        # print(f"    ✗ Splitting does not improve fit: increase of {avg_new_distance:.2f} >= {original_distance:.2f}")
        return None


def split_multiple_cell_layers(cell_1, cell_2):
    z_to_predicts = [1, 2]
    split_cells = 0
    for z_to_predict in z_to_predicts:
        # Split cell_1 at its top
        # print("bottom cell z:", z_to_predict)
        split_results = split_single_cell_layer(
            cell_1, cell_2, z_to_predict=z_to_predict, bottom_up=True
        )
        if split_results is not None:
            split_cells += 1
            part_1_coords, part_2_coords, z_idx, last_z = split_results
            part_1_coords_2d = set(map(tuple, part_1_coords))
            part_2_coords_2d = set(map(tuple, part_2_coords))
            update_coords(
                cell_1, cell_2, z_idx, part_1_coords_2d, part_2_coords_2d, last_z
            )

        # print("top cell z:", z_to_predict)
        # Split cell_2 at its bottom
        split_results = split_single_cell_layer(
            cell_2, cell_1, z_to_predict=z_to_predict, bottom_up=False
        )
        if split_results is not None:
            split_cells += 1
            part_1_coords, part_2_coords, z_idx, first_z = split_results
            part_1_coords_2d = set(map(tuple, part_1_coords))
            part_2_coords_2d = set(map(tuple, part_2_coords))
            update_coords_bottom(
                cell_2, cell_1, z_idx, part_1_coords_2d, part_2_coords_2d, first_z
            )
        
        # When no splits are made on the 1st level, we dont need to check splitting the second level.
        if split_cells == 0:
            break
    return split_cells


def split_single_cell_layer(cell_1, cell_2, z_to_predict=1, bottom_up=True):
    """
    Split a cell layer at the touching interface.
    bottom_up=True: split cell_1 at its top (for cell_1 splitting cell_2 from above)
    bottum_up=False: split cell_1 at its bottom (for cell_1 splitting from below)
    """
    if len(cell_1.volumes_2d) <= 2 or len(cell_2.volumes_2d) <= 2:
        return None

    z_positions_1, y_positions_1, x_positions_1 = get_centroid_positions(
        cell_1.centroids_2d
    )
    z_positions_2, y_positions_2, x_positions_2 = get_centroid_positions(
        cell_2.centroids_2d
    )

    # Get line fits
    if bottom_up:
        line_1 = calculate_line_3d(
            z_positions_1, y_positions_1, x_positions_1, points=(0, -z_to_predict)
        )
        line_2 = calculate_line_3d(
            z_positions_2[::-1],
            y_positions_2[::-1],
            x_positions_2[::-1],
            points=(0, -z_to_predict),
        )
        z_idx = len(cell_1.volumes_2d) - z_to_predict
        target_z = z_positions_1[-z_to_predict]
    else:
        line_1 = calculate_line_3d(
            z_positions_1[::-1], y_positions_1[::-1], x_positions_1[::-1], points=(0, -z_to_predict)
        )
        line_2 = calculate_line_3d(
            z_positions_2, y_positions_2, x_positions_2, points=(0, -z_to_predict)
        )
        z_idx = z_to_predict - 1
        target_z = z_positions_1[z_to_predict - 1]

    # Get predictions
    predict_x_1, predict_y_1 = line_1(target_z)
    predict_x_2, predict_y_2 = line_2(target_z)

    # Check if predicted points are in cell_1's coords_set
    predicted_line_1_in_cell_1 = (
        int(round(target_z)),
        int(round(predict_y_1)),
        int(round(predict_x_1)),
    ) in cell_1.coords_set
    predicted_line_2_in_cell_1 = (
        int(round(target_z)),
        int(round(predict_y_2)),
        int(round(predict_x_2)),
    ) in cell_1.coords_set

    # print(predicted_line_1_in_cell_1, predicted_line_2_in_cell_1)
    # print(int(round(predict_y_1*0.64)), int(round(predict_x_1*0.64)), int(round(predict_y_2*0.64)), int(round(predict_x_2*0.64)))
    if not (predicted_line_1_in_cell_1 and predicted_line_2_in_cell_1):
        # print(
        #     f"  ✗ Predicted splitting points not in cell {cell_1.label} at z={int(round(target_z))}"
        # )
        return None

    pred_point_1 = np.array([predict_y_1, predict_x_1])
    pred_point_2 = np.array([predict_y_2, predict_x_2])

    # print(f"Attempting to split touching cells {cell_1.label} and {cell_2.label}")
    split_results = check_split_cells(cell_1, z_idx, pred_point_1, pred_point_2)
    if split_results is None:
        return None

    part_1_coords, part_2_coords = split_results
    target_z_int = int(round(target_z))
    return part_1_coords, part_2_coords, z_idx, target_z_int


def update_coords(cell_1, cell_2, z_idx, part_1_coords_2d, part_2_coords_2d, last_z):
    """Update both cells after splitting cell_1 at its top"""
    cell_1.coords_set_2d[z_idx] = part_1_coords_2d

    for y, x in part_2_coords_2d:
        cell_1.coords_set.discard((last_z, int(y), int(x)))

    cell_1.volumes_2d[z_idx] = len(part_1_coords_2d)
    part_1_array = np.array(list(part_1_coords_2d))
    cell_1.centroids_2d[z_idx] = (last_z, *np.mean(part_1_array, axis=0))

    # Append to cell_2
    cell_2.coords_set_2d.append(part_2_coords_2d)
    cell_2.labels_2d.append(cell_1.labels_2d[z_idx])

    part_2_array = np.array(list(part_2_coords_2d))
    cell_2.centroids_2d.append((last_z, *np.mean(part_2_array, axis=0)))
    cell_2.volumes_2d.append(len(part_2_coords_2d))

    for y, x in part_2_coords_2d:
        cell_2.coords_set.add((last_z, int(y), int(x)))

    _recalculate_cell_properties(cell_1)
    _recalculate_cell_properties(cell_2)


def update_coords_bottom(
    cell_2, cell_1, z_idx, part_1_coords_2d, part_2_coords_2d, first_z
):
    """Update both cells after splitting cell_2 at its bottom"""
    cell_2.coords_set_2d[z_idx] = part_1_coords_2d

    for y, x in part_2_coords_2d:
        cell_2.coords_set.discard((first_z, int(y), int(x)))

    cell_2.volumes_2d[z_idx] = len(part_1_coords_2d)
    part_1_array = np.array(list(part_1_coords_2d))
    cell_2.centroids_2d[z_idx] = (first_z, *np.mean(part_1_array, axis=0))

    # Insert at beginning of cell_1
    cell_1.coords_set_2d.insert(0, part_2_coords_2d)
    cell_1.labels_2d.insert(0, cell_2.labels_2d[z_idx])

    part_2_array = np.array(list(part_2_coords_2d))
    cell_1.centroids_2d.insert(0, (first_z, *np.mean(part_2_array, axis=0)))
    cell_1.volumes_2d.insert(0, len(part_2_coords_2d))

    for y, x in part_2_coords_2d:
        cell_1.coords_set.add((first_z, int(y), int(x)))

    _recalculate_cell_properties(cell_1)
    _recalculate_cell_properties(cell_2)


def _recalculate_cell_properties(cell):
    """Helper to recalculate volume and centroid after modification"""
    cell.volume = sum(cell.volumes_2d)
    cell.centroid = np.mean(cell.centroids_2d, axis=0)


import copy

def split_organoid_cells(organoid):
    print("Starting to split touching cells...")
    organoid = find_touching_pairs(organoid)
    touching_cells = find_touching_cells(organoid)
    print(f"Found {len(touching_cells)} pairs of touching cells.")
    number_of_splits = 0
    for pair in touching_cells:
        cell_1 = next(c for c in organoid if c.label == pair[0])
        cell_2 = next(c for c in organoid if c.label == pair[1])
        # Ensure cell_1 is below cell_2 (lower z centroid)
        if cell_1.centroid[0] > cell_2.centroid[0]:
            cell_1, cell_2 = cell_2, cell_1
        number_of_splits += split_multiple_cell_layers(cell_1, cell_2)
    
    print(f"Splitted {number_of_splits} touching cell pairs.")
    return organoid


def organoid_from_mask(mask):
    """Reconstruct organoid list directly from mask by splitting coords by z-level"""
    props = regionprops(mask)

    organoid = []
    for prop in props:
        # Get 3D coordinates (z, y, x)
        coords_3d = prop.coords

        # Split coordinates by z-level
        coords_by_z = {}
        for z, y, x in coords_3d:
            if z not in coords_by_z:
                coords_by_z[z] = []
            coords_by_z[z].append((y, x))

        # Sort by z to maintain order
        z_sorted = sorted(coords_by_z.keys())

        # Build per-slice data
        centroids_2d = []
        coords_set_2d = []
        volumes_2d = []

        for z in z_sorted:
            coords_2d = np.array(coords_by_z[z])
            centroid_2d = np.mean(coords_2d, axis=0)

            centroids_2d.append((z, centroid_2d[0], centroid_2d[1]))
            coords_set_2d.append(set(map(tuple, coords_2d)))
            volumes_2d.append(len(coords_2d))

        cell = cell_3d(
            label=prop.label,
            centroid=prop.centroid,
            coords_set=set(map(tuple, prop.coords)),
            volume=prop.area,
            centroids_2d=centroids_2d,
            coords_set_2d=coords_set_2d,
            volumes_2d=volumes_2d,
        )

        organoid.append(cell)

    return organoid


def stitch_small_cells(organoid, max_slices=2, iou_threshold=0.9):
    """Merge small cells (1-2 slices) with neighboring cells above or below"""
    cells_to_remove = []
    number_of_cells_stitched = 0
    number_of_cells_removed = 0

    print(f"Stitching cells with {max_slices} or fewer slices...")

    for cell in organoid:
        if len(cell.volumes_2d) > max_slices:
            continue  # Only process cells with max_slices or fewer

        # Get the z positions of this cell's slices
        z_positions = [int(round(c[0])) for c in cell.centroids_2d]

        # Find neighboring cells in organoid that are above or below
        candidates = []
        for c in organoid:
            if c.label == cell.label or len(c.volumes_2d) <= max_slices:
                continue

            # Check if candidate cell is adjacent to any of the current cell's slices
            for z_pos in z_positions:
                # Check adjacent z-levels
                for neighbor_z in [z_pos - 1, z_pos + 1]:
                    # Get the slice index in the candidate cell at neighbor_z
                    candidate_z_positions = [
                        int(round(c_cent[0])) for c_cent in c.centroids_2d
                    ]

                    if neighbor_z in candidate_z_positions:
                        # Found an adjacent slice, now check IOU
                        candidate_slice_idx = candidate_z_positions.index(neighbor_z)
                        cell_slice_idx = z_positions.index(z_pos)

                        iou = calculate_iou(
                            {
                                0: cell.coords_set_2d[cell_slice_idx],
                                1: c.coords_set_2d[candidate_slice_idx],
                            },
                            0,
                            1,
                        )

                        if iou > iou_threshold:
                            candidates.append(c)
                            break  # Found a valid neighbor, no need to check other slices of this candidate

                if c in candidates:
                    break  # Already added this candidate

        if not candidates:
            # Only remove 1-slice cells, keep 2-slice cells
            if len(cell.volumes_2d) == 1:
                cells_to_remove.append(cell.label)
                number_of_cells_removed += 1
            continue

        number_of_cells_stitched += 1
        closest_neighbor = candidates[0]

        # Add all of cell's slices to neighbor
        for i in range(len(cell.coords_set_2d)):
            # Find correct insertion position to maintain z-order
            cell_z = int(round(cell.centroids_2d[i][0]))
            neighbor_z_positions = [
                int(round(c[0])) for c in closest_neighbor.centroids_2d
            ]

            # Find insertion index
            insert_idx = len(neighbor_z_positions)
            for idx, neighbor_z in enumerate(neighbor_z_positions):
                if cell_z < neighbor_z:
                    insert_idx = idx
                    break

            # Insert at correct position
            closest_neighbor.coords_set_2d.insert(insert_idx, cell.coords_set_2d[i])
            closest_neighbor.centroids_2d.insert(insert_idx, cell.centroids_2d[i])
            closest_neighbor.volumes_2d.insert(insert_idx, cell.volumes_2d[i])

            # Update 3D coords_set
            closest_neighbor.coords_set.update(
                {(cell_z, int(y), int(x)) for y, x in cell.coords_set_2d[i]}
            )

        # Update neighbor properties
        _recalculate_cell_properties(closest_neighbor)

        # Mark cell for removal
        cells_to_remove.append(cell.label)

    # Remove merged cells from organoid
    organoid = [c for c in organoid if c.label not in cells_to_remove]

    print(
        f"Stitched {number_of_cells_stitched} small cells, and removed {number_of_cells_removed} cells without suitable neighbors."
    )

    return organoid

import copy

def stitch_3d(mask, image1, image_type_1=None, image2=None, image_type_2=None, breaking_threshold=2.5):
    print("Starting 3D stitching process...")
    organoid = get_2d_mask_properties(mask, image1, image_type_1=image_type_1, image2=image2, image_type_2=image_type_2)
    print(f"Initial organoid has {len(organoid)} 2d sliced cells.")
    organoid = IOU_stitching(organoid, distance_threshold=10, iou_threshold=0.3)

    # Iterative break and restitching
    max_iterations = 10
    iteration = 0
    print("Starting iterative breaking of cells")
    while iteration < max_iterations:
        organoid = get_break_scores(organoid)
        organoid, breaks_made = break_stitching(organoid, breaking_threshold=breaking_threshold)
        if breaks_made == 0:
            break  # No more breaks made

        print(f"    Iteration {iteration + 1}: Made {breaks_made} breaks.")
        iteration += 1

    # before_splitting = copy.deepcopy(organoid)
    # before_splitting_mask = mask_from_organoid(before_splitting, shape=mask.shape)
    organoid = split_organoid_cells(organoid)

    stitched_mask = mask_from_organoid(organoid, shape=mask.shape)
    organoid = organoid_from_mask(stitched_mask)

    organoid = stitch_small_cells(organoid)

    stitched_mask = mask_from_organoid(organoid, shape=mask.shape)
    

    print(f"Final stitched mask has {len(organoid)} cells.")

    return stitched_mask, organoid #, before_splitting, before_splitting_mask

In [8]:
import time
import napari


output_directory = r"C:\Users\6331823\Local SSD\Fake_data"
image = tifffile.imread(os.path.join(output_directory, "realistic_organoid.tif"))
mask = tifffile.imread(
    os.path.join(output_directory, "realistic_organoid_segmented.tif")
)
perfect_mask = tifffile.imread(
    os.path.join(output_directory, "perfect_mask.tif")
)

# organoid = get_2d_mask_properties(mask, image, image_type_1="wt")
# organoid = IOU_stitching(organoid, distance_threshold=10, iou_threshold=0.3)
# stitched_mask = mask_from_organoid(organoid, mask.shape)
start_time = time.time()    
stitched_mask, organoid, before_splitting_mask = stitch_3d(
    mask,
    image,
    image_type_1="wt",
    breaking_threshold=2,
)
end_time = time.time()
print(f"Stitching completed in {end_time - start_time:.2f} seconds.")

# tifffile.imwrite(
#     os.path.join(output_directory, "3d_segmented.tif"), stitched_mask
# )

# print(centroid_accuracy(perfect_mask, stitched_mask))
# print(f"actual amount of cells: {len(np.unique(perfect_mask)) - 1}, predicted amount of cells: {len(np.unique(stitched_mask)) - 1}")
scale = (2.5, 0.64, 0.64)
viewer = napari.Viewer()
viewer.add_image(image, name="wt image", blending="additive", scale=scale)
viewer.add_labels(mask, name="original mask", visible=False, scale=scale)
viewer.add_labels(stitched_mask, name="stitched mask", visible=False, scale=scale)
viewer.add_labels(perfect_mask, name="perfect mask", visible=False, scale=scale)
viewer.add_labels(before_splitting_mask, name="before split mask", visible=False, scale=scale)

Starting 3D stitching process...
Initial organoid has 1864 2d sliced cells.
Starting IOU based 3D stitching...
Completed IOU based stitching: 178 cells formed.
Starting iterative breaking of cells
    Iteration 1: Made 20 breaks.
    Iteration 2: Made 9 breaks.
    Iteration 3: Made 1 breaks.
Starting to split touching cells...
Found 108 pairs of touching cells.
Splitted 20 touching cell pairs.
Stitching cells with 2 or fewer slices...
Stitched 7 small cells, and removed 1 cells without suitable neighbors.
Final stitched mask has 200 cells.
Stitching completed in 4.01 seconds.


<Labels layer 'before split mask' at 0x1d7018cd350>

In [10]:
output_directory = r"C:\Users\6331823\Local SSD\Data_Merel\Image optimisation\Plate14_20x_2x2bin_2.5um_F0"
image = tifffile.imread(os.path.join(output_directory, "cropped", "Frame-0.tif"))
stitched_mask = tifffile.imread(
    os.path.join(output_directory, "segmented", "Frame-0_masks.tif")
)


scale = (2.5, 0.64, 0.64)
viewer = napari.Viewer()
viewer.add_image(image[:, 0], name="wt image", blending="additive", scale=scale)
viewer.add_labels(stitched_mask, name="stitched mask", visible=False, scale=scale)

<Labels layer 'stitched mask' at 0x245989b5750>

In [6]:
import time
output_directory = r"C:\Users\6331823\Local SSD\Data_Merel\MvL_Exp139_positions_WT_SI\Experiment_final_1_F02"
image = tifffile.imread(os.path.join(output_directory, "cropped", "Frame-20.tif"))[:,:,:,:606]
mask = tifffile.imread(
    os.path.join(output_directory, "segmented", "Frame-20_masks.tif")
)
# perfect_mask = tifffile.imread(os.path.join(output_directory, "perfect_mask.tif"))

start_time = time.time()
stitched_mask, organoid = stitch_3d(
    mask,
    image1=image[:,0],
    image_type_1="crc",
    image2=image[:,1],
    image_type_2="wt",
    breaking_threshold=2.5,
)
end_time = time.time()
print(f"Stitching completed in {end_time - start_time:.2f} seconds.")

# tifffile.imwrite(
#     os.path.join(output_directory, "3d_segmented.tif"), stitched_mask
# )

# print(centroid_accuracy(perfect_mask, stitched_mask))
# print(f"actual amount of cells: {len(np.unique(perfect_mask)) - 1}, predicted amount of cells: {len(np.unique(stitched_mask)) - 1}")
scale = (2.5, 0.64, 0.64)
viewer = napari.Viewer()
viewer.add_image(image[:,0], name="wt image", blending="additive", scale=scale)
viewer.add_image(image[:,1], name="crc image", blending="additive", scale=scale)
viewer.add_labels(mask, name="original mask", visible=False, scale=scale)
viewer.add_labels(stitched_mask, name="stitched mask", visible=False, scale=scale)
# viewer.add_labels(perfect_mask, name="perfect mask", visible=False, scale=scale)
# viewer.add_labels(
#     before_splitting_mask, name="before split mask", visible=False, scale=scale
# )

Starting 3D stitching process...
Initial organoid has 448 2d sliced cells.
Starting IOU based 3D stitching...
Completed IOU based stitching: 213 cells formed.
Starting iterative breaking of cells
    Iteration 1: Made 3 breaks.
Starting to split touching cells...
Found 145 pairs of touching cells.
Splitted 7 touching cell pairs.
Stitching cells with 2 or fewer slices...
Stitched 1 small cells, and removed 90 cells without suitable neighbors.
Final stitched mask has 125 cells.
Stitching completed in 0.23 seconds.


<Labels layer 'stitched mask' at 0x2457f8ca9d0>

Suijkerbuijk segmentation

In [ ]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))
import utils
import time

input_directory = r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Drosophila_1"
# input_directory = r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Zebrafish_1"
# input_directory = r"C:\Users\6331823\Local SSD\Fake_data"
# input_directory = r"C:\Users\6331823\Downloads\Data for mario\Merel organoid"
# input_directory = r"C:\Users\6331823\Local SSD\Data_Merel\MvL_Exp139_positions_WT_SI\Experiment_final_1_F18"

image = tifffile.imread(os.path.join(input_directory, "data.tif"))

# model = utils.load_model(
#     r"C:\Users\6331823\Local SSD\Train dual nuclei 20x\models\dual_nuclei_wt"
# )
model = utils.load_model(r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Drosophila_1\model\drosophila")
# model = utils.load_model(
#     r"C:\Users\6331823\Local SSD\Train fixed intestinal 40x\models\fixed_intestinal_40x"
# )
# model = utils.load_model(
#     r"C:\Users\6331823\Local SSD\Train model cell segmentation\models\cell_segmentation_4"
# )

start_time = time.time()
# mask = utils.segment(image, model)
# tifffile.imwrite(
#     os.path.join(input_directory, "2d_segmented_drosophila_model.tif"),
#     mask,
#     compression="zlib",
#     compressionargs={"level": 8},
# )
mask = tifffile.imread(os.path.join(input_directory, "2d_segmented_drosophila_model.tif"))
segmentation_time = time.time() - start_time


start_stitch_time = time.time()
stitched_mask, organoid = stitch_3d(
    mask,
    image,
    image_type_1="wt",
    breaking_threshold=2,
)
end_time = time.time()

tifffile.imwrite(
    os.path.join(input_directory, "3d_segmented_suijkerbuijk_drosophila_model.tif"),
    stitched_mask,
    compression="zlib",
    compressionargs={"level": 8},
)

# ground_truth = tifffile.imread(
#     os.path.join(input_directory, "GroundTruth.tif")
# )

# results = (centroid_accuracy(ground_truth, stitched_mask))
# print(f"centroid accuracy: {results}")
# print(
#     f"actual amount of cells: {len(np.unique(ground_truth)) - 1}, predicted amount of cells: {len(np.unique(stitched_mask)) - 1}"
# )

print(f"Segmentation completed in {segmentation_time:.2f} seconds.")
print(f"Stitching completed in {end_time - start_stitch_time:.2f} seconds.")
print(f"Total time: {end_time - start_time:.2f} seconds.")

# save results of centroid accuracy and cell count to a text file
# with open(os.path.join(input_directory, "results.txt"), "w") as f:
#     f.write(f"centroid accuracy: {results}\n")
#     f.write(
#         f"actual amount of cells: {len(np.unique(ground_truth)) - 1}, predicted amount of cells: {len(np.unique(stitched_mask)) - 1}\n"
#     )
#     f.write(f"Segmentation time: {segmentation_time:.2f} seconds.\n")
#     f.write(f"Stitching time: {end_time - start_stitch_time:.2f} seconds.\n")
#     f.write(f"Total time: {end_time - start_time:.2f} seconds.\n")

Using CUDA for processing.
2026-02-04 14:16:41,186 [INFO] ** TORCH CUDA version installed and working. **
2026-02-04 14:16:41,186 [INFO] >>>> using GPU (CUDA)
2026-02-04 14:16:41,186 [WARNING] pretrained model C:\Users\6331823\.cellpose\models\cpsam not found, using default model
2026-02-04 14:16:42,323 [INFO] >>>> loading model C:\Users\6331823\.cellpose\models\cpsam
loaded custom model: drosophila
Starting 3D stitching process...
Initial organoid has 11261 2d sliced cells.
Starting IOU based 3D stitching...
Completed IOU based stitching: 1274 cells formed.
Starting iterative breaking of cells


TypeError: 'numpy.int32' object is not callable

Cellpose 4

In [ ]:
import os
import time
import sys
from cellpose import models, io

# input_directory = r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Drosophila_1"
# input_directory = r"C:\Users\6331823\Local SSD\Fake_data"
# input_directory = r"C:\Users\6331823\Downloads\Data for mario\Merel organoid"
input_directory = r"C:\Users\6331823\Local SSD\Data_Merel\MvL_Exp139_positions_WT_SI\Experiment_final_1_F18"

image = tifffile.imread(os.path.join(input_directory, "data_2.tif"))

# model = models.CellposeModel(gpu=True)
sys.path.insert(0, os.path.dirname(os.getcwd()))
import utils
model = utils.load_model(
    r"C:\Users\6331823\Local SSD\Train dual nuclei 20x\models\dual_nuclei_wt"
)
# model = utils.load_model(
#     r"C:\Users\6331823\Local SSD\Train fixed intestinal 40x\models\fixed_intestinal_40x"
# )
start_time = time.time()

stitched_mask, _, _ = model.eval(
            image,
            diameter=None,
            normalize=True,
            invert=False,
            resample=True,
            do_3D=True,
            flow3D_smooth=0.4,
            progress=None,
            z_axis=0,
            anisotropy=1,
            min_size=15,
        )

segmentation_time = time.time() - start_time
end_time = time.time()

tifffile.imwrite(
    os.path.join(input_directory, "3d_segmented_cellpose4.tif"),
    stitched_mask
)

# ground_truth = tifffile.imread(os.path.join(input_directory, "GroundTruth.tif"))

# results = centroid_accuracy(ground_truth, stitched_mask)
# print(f"centroid accuracy: {results}")
# print(
#     f"actual amount of cells: {len(np.unique(ground_truth)) - 1}, predicted amount of cells: {len(np.unique(stitched_mask)) - 1}"
# )

print(f"Segmentation completed in {segmentation_time:.2f} seconds.")
print(f"Total time: {end_time - start_time:.2f} seconds.")

# # save results of centroid accuracy and cell count to a text file
# with open(os.path.join(input_directory, "results_cellpose4.txt"), "w") as f:
#     f.write(f"centroid accuracy: {results}\n")
#     f.write(
#         f"actual amount of cells: {len(np.unique(ground_truth)) - 1}, predicted amount of cells: {len(np.unique(stitched_mask)) - 1}\n"
#     )
#     f.write(f"Segmentation time: {segmentation_time:.2f} seconds.\n")
#     f.write(f"Stitching time: {end_time - start_stitch_time:.2f} seconds.\n")
#     f.write(f"Total time: {end_time - start_time:.2f} seconds.\n")

Using CUDA for processing.
loaded custom model: dual_nuclei
Segmentation completed in 41.79 seconds.
Total time: 41.79 seconds.


In [22]:
print(f"predicted amount of cells: {len(np.unique(stitched_mask)) - 1}")

predicted amount of cells: 8755


In [3]:
viewer = napari.Viewer()
# input_directory = r"C:\Users\6331823\Downloads\Data for mario\Merel organoid"
input_directory = r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Drosophila_1"
# input_directory = r"C:\Users\6331823\Local SSD\Data_Merel\MvL_Exp139_positions_WT_SI\Experiment_final_1_F18"
image = tifffile.imread(os.path.join(input_directory, "data_3.tif"))
suijkerbuijk = tifffile.imread(
    os.path.join(input_directory, "3d_segmented_suijkerbuijk.tif")
)
cellpose4 = tifffile.imread(
    os.path.join(input_directory, "3d_segmented_cellpose4.tif")
)
# ground_truth = tifffile.imread(
#     os.path.join(input_directory, "GroundTruth.tif")
# )

scale = (2.5, 0.32, 0.32)
viewer.add_image(image, name="wt image", blending="additive", scale=scale)
viewer.add_labels(suijkerbuijk, name="suijkerbuijk mask", visible=False, scale=scale)
viewer.add_labels(cellpose4, name="cellpose4 mask", visible=False, scale=scale)
# viewer.add_labels(ground_truth, name="ground truth", visible=False, scale=scale)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\6331823\\Downloads\\NIS3D\\NIS3D\\NIS3D\\Drosophila_1\\data_3.tif'

In [14]:
print(f"predicted amount of cells: {len(np.unique(suijkerbuijk)) - 1}")

predicted amount of cells: 855


In [55]:
import os
import tifffile

def get_segmentation_score(input_directory):
    # ground_truth = tifffile.imread(
    #     os.path.join(input_directory, "GroundTruth.tif")
    # )
    ground_truth = tifffile.imread(
        os.path.join(input_directory, "perfect_mask.tif")
    )
    stitched_mask_suijkerbuijk = tifffile.imread(
        os.path.join(input_directory, "3d_segmented.tif")
    )
    image = tifffile.imread(os.path.join(input_directory, "realistic_organoid.tif"))
    stitched_mask_cp4 = tifffile.imread(os.path.join(input_directory, "realistic_organoid_cp_masks.tif"))
    results_cp4 = centroid_accuracy(ground_truth, stitched_mask_cp4)
    results_suijkerbuijk = centroid_accuracy(ground_truth, stitched_mask_suijkerbuijk)
    cell_count_info = (
        f"actual amount of cells: {len(np.unique(ground_truth)) - 1}, predicted amount of cells: {len(np.unique(stitched_mask_suijkerbuijk)) - 1}"
    )

    scale = (2.5, 0.64, 0.64)
    viewer = napari.Viewer()
    viewer.add_image(image, name="wt image", blending="additive", scale=scale)
    # viewer.add_labels(ground_truth, name="perfect mask", visible=False)
    viewer.add_labels(stitched_mask_cp4, name="cp4 stitched mask", visible=False, scale=scale)
    viewer.add_labels(stitched_mask_suijkerbuijk, name="suijkerbuijk stitched mask", visible=False, scale=scale)
    return results_cp4, results_suijkerbuijk, cell_count_info

# input_directory = r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Zebrafish_1"
input_directory = r"C:\Users\6331823\Local SSD\Fake_data"

results_cp4, results_suijkerbuijk, cell_count_info = get_segmentation_score(input_directory)
print(results_cp4)
print(results_suijkerbuijk)
print(cell_count_info)

{'true_positives': 200, 'false_negatives': 0, 'false_positives': 3, 'merged_cells': 0, 'precision': 0.9852216748768473, 'recall': 1.0, 'f1': 0.9925558312655086, 'voxel_dice': 0.6218564047190718, 'false_positive_labels': [201, 202, 203]}
{'true_positives': 200, 'false_negatives': 0, 'false_positives': 0, 'merged_cells': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'voxel_dice': 0.5385429021272464, 'false_positive_labels': []}
actual amount of cells: 200, predicted amount of cells: 200


In [5]:
import napari
import tifffile
import os

# input_directory = r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Drosophila_1"
input_directory = r"C:\Users\6331823\Local SSD\Fake_data"

image = tifffile.imread(os.path.join(input_directory, "data.tif"))
stitched_mask = tifffile.imread(os.path.join(input_directory, "3d_segmented.tif"))

ground_truth = tifffile.imread(os.path.join(input_directory, "GroundTruth.tif"))
viewer = napari.Viewer()
# scale = (1,1,1)
scale = (2.5, 0.62, 0.62)
viewer.add_image(image, name="zebrafish 1 image", blending="additive", scale=scale)
viewer.add_labels(stitched_mask, name="3d stitched mask", visible=False, scale=scale)
viewer.add_labels(ground_truth, name="ground truth", visible=False, scale=scale)

<Labels layer 'ground truth' at 0x292932a7f10>

In [3]:
from functools import lru_cache
import time
import tifffile

def weighted_metrics(ground_truth, prediction, confidence_map=None):
    """
    Calculate weighted precision, recall, F1, IoU, and SEG scores (memory-efficient version).
    """
    start_total = time.time()

    ground_truth = np.asarray(ground_truth, dtype=np.uint32)
    prediction = np.asarray(prediction, dtype=np.uint32)

    if ground_truth.shape != prediction.shape:
        raise ValueError("Masks must share shape")

    if confidence_map is not None:
        confidence_map = np.asarray(confidence_map, dtype=np.uint8)
        if confidence_map.shape != ground_truth.shape:
            raise ValueError("Confidence map must have same shape as masks")

    # Get unique labels (excluding background)
    gt_labels = np.unique(ground_truth)
    gt_labels = gt_labels[gt_labels > 0]
    pred_labels = np.unique(prediction)
    pred_labels = pred_labels[pred_labels > 0]

    print(
        f"Processing {len(gt_labels)} GT cells and {len(pred_labels)} predicted cells..."
    )

    # Build confidence scores
    confidence_scores = {}
    start_conf = time.time()

    if confidence_map is not None:
        conf_value_to_score = np.array([0.0, 0.0, 1 / 3, 2 / 3, 1.0], dtype=np.float32)

        for gt_label in gt_labels:
            gt_mask = ground_truth == gt_label
            cell_conf_values = confidence_map[gt_mask]
            undefined_ratio = np.sum(cell_conf_values == 1) / len(cell_conf_values)

            if undefined_ratio > 0.5:
                confidence_scores[gt_label] = 0.0
                continue

            valid_mask = cell_conf_values != 1
            if np.any(valid_mask):
                valid_pixels = cell_conf_values[valid_mask]
                weighted_conf = np.mean(conf_value_to_score[valid_pixels])
                confidence_scores[gt_label] = float(weighted_conf)
            else:
                confidence_scores[gt_label] = 0.0
    else:
        for gt_label in gt_labels:
            confidence_scores[gt_label] = 1.0

    valid_gt_labels = np.array(
        [l for l in gt_labels if confidence_scores.get(l, 0) > 0]
    )
    print(
        f"  Confidence calculation done: {time.time() - start_conf:.2f}s ({len(valid_gt_labels)} valid GT cells)"
    )

    # Compute centroids WITHOUT storing full masks
    start_spatial = time.time()
    print(f"  Computing centroids and IoU for nearby cells...")

    gt_centroids = {}
    pred_centroids = {}

    # Only compute centroids for valid GT labels
    for gt_label in valid_gt_labels:
        coords = np.where(ground_truth == gt_label)
        gt_centroids[int(gt_label)] = np.array(
            [np.mean(coords[0]), np.mean(coords[1]), np.mean(coords[2])]
        )

    for pred_label in pred_labels:
        coords = np.where(prediction == pred_label)
        pred_centroids[int(pred_label)] = np.array(
            [np.mean(coords[0]), np.mean(coords[1]), np.mean(coords[2])]
        )

    # Build KDTree for fast nearest neighbor queries
    from scipy.spatial import cKDTree

    gt_centroids_array = np.array([gt_centroids[int(l)] for l in valid_gt_labels])
    pred_centroids_array = np.array([pred_centroids[int(l)] for l in pred_labels])

    pred_tree = cKDTree(pred_centroids_array)

    # Only compute IoU for nearby cells - compute masks on-the-fly
    max_distance = 100
    k_neighbors = 5

    iou_matrix = {}
    computed_ious = 0

    for i, gt_label in enumerate(valid_gt_labels):
        gt_label = int(gt_label)

        # Find nearby prediction cells
        distances, indices = pred_tree.query(
            gt_centroids_array[i], k=min(k_neighbors, len(pred_labels))
        )

        iou_row = {}

        # Create GT mask once per cell
        gt_mask = ground_truth == gt_label

        for idx, dist in zip(indices, distances):
            # Skip if too far away
            if dist > max_distance:
                continue

            pred_label = int(pred_labels[idx])

            # Create pred mask on-the-fly (small memory footprint)
            pred_mask = prediction == pred_label

            intersection = np.count_nonzero(gt_mask & pred_mask)
            if intersection == 0:
                continue  # Skip non-overlapping pairs

            union = np.count_nonzero(gt_mask | pred_mask)
            iou_row[pred_label] = intersection / union if union > 0 else 0.0
            computed_ious += 1

            # Delete pred_mask immediately to free memory
            del pred_mask

        del gt_mask  # Free GT mask for this cell
        iou_matrix[gt_label] = iou_row

    print(
        f"  IoU calculation done: {time.time() - start_spatial:.2f}s (computed {computed_ious} IoU pairs)"
    )

    # Matching
    start_match = time.time()
    print(f"  Finding best matches...")
    gt_matches = {}
    matched_pred_labels = set()

    for gt_label in valid_gt_labels:
        gt_label = int(gt_label)
        iou_row = iou_matrix.get(gt_label, {})

        if not iou_row:
            gt_matches[gt_label] = (None, 0.0)
            continue

        best_pred_label, best_iou = max(iou_row.items(), key=lambda x: x[1])

        # Check bidirectional match
        pred_best_gt_score = 0
        pred_best_gt = None

        for other_gt in valid_gt_labels:
            other_gt = int(other_gt)
            if best_pred_label in iou_matrix.get(other_gt, {}):
                score = iou_matrix[other_gt][best_pred_label]
                if score > pred_best_gt_score:
                    pred_best_gt_score = score
                    pred_best_gt = other_gt

        if pred_best_gt == gt_label:
            gt_matches[gt_label] = (best_pred_label, best_iou)
            matched_pred_labels.add(best_pred_label)
        else:
            gt_matches[gt_label] = (None, 0.0)

    print(
        f"  Matching done: {time.time() - start_match:.2f}s ({len(matched_pred_labels)} matches found)"
    )

    # Calculate metrics (final pass - compute on-the-fly)
    start_metrics = time.time()
    W_TP = 0.0
    W_FN = 0.0
    weighted_seg_numerator = 0.0

    for gt_label in valid_gt_labels:
        gt_label = int(gt_label)
        confidence = confidence_scores.get(gt_label, 1.0)
        matched_pred, iou_score = gt_matches[gt_label]

        if matched_pred is not None:
            W_TP += confidence
            weighted_seg_numerator += confidence * iou_score
        else:
            W_FN += confidence

    FP = len(pred_labels) - len(matched_pred_labels)

    W_Precision = W_TP / (W_TP + FP) if (W_TP + FP) > 0 else 0.0
    W_Recall = W_TP / (W_TP + W_FN) if (W_TP + W_FN) > 0 else 0.0
    W_F1 = (2 * W_TP) / (2 * W_TP + FP + W_FN) if (2 * W_TP + FP + W_FN) > 0 else 0.0

    # Compute binary metrics on-the-fly (no large intermediate arrays)
    gt_binary = ground_truth > 0
    pred_binary = prediction > 0
    intersection_mask = gt_binary & pred_binary

    # Recompute confidence weight map only for pixels needed
    if confidence_map is not None:
        conf_value_to_score = np.array([0.0, 0.0, 1 / 3, 2 / 3, 1.0], dtype=np.float32)
        W_IoU_numerator = 0.0
        for gt_label in valid_gt_labels:
            mask = (ground_truth == gt_label) & intersection_mask
            if np.any(mask):
                conf_vals = conf_value_to_score[confidence_map[mask]]
                W_IoU_numerator += len(mask) * np.mean(conf_vals)
    else:
        W_IoU_numerator = np.count_nonzero(intersection_mask)

    W_IoU_denominator = np.count_nonzero(gt_binary) + np.count_nonzero(
        pred_binary & ~gt_binary
    )
    W_IoU = W_IoU_numerator / W_IoU_denominator if W_IoU_denominator > 0 else 0.0

    W_SEG = (
        weighted_seg_numerator / (W_TP + W_FN + FP) if (W_TP + W_FN + FP) > 0 else 0.0
    )

    print(f"  Metrics calculation done: {time.time() - start_metrics:.2f}s")
    print(f"Total time: {time.time() - start_total:.2f}s\n")

    return {
        "W-TP": W_TP,
        "W-FN": W_FN,
        "FP": FP,
        "W-Precision": W_Precision,
        "W-Recall": W_Recall,
        "W-F1": W_F1,
        "W-IoU": W_IoU,
        "W-SEG": W_SEG,
        "num_gt_cells": len(valid_gt_labels),
        "num_pred_cells": len(pred_labels),
        "num_matched": len(matched_pred_labels),
    }


input_directory = r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Drosophila_1"

confidence_map = tifffile.imread(
    os.path.join(input_directory, "ConfidenceScore.tif")
)

stitched_mask = tifffile.imread(os.path.join(input_directory, "3d_segmented.tif"))

ground_truth = tifffile.imread(os.path.join(input_directory, "GroundTruth.tif"))

# Calculate weighted metrics
results = weighted_metrics(ground_truth, stitched_mask, confidence_map)
print(f"W-Precision: {results['W-Precision']:.3f}")
print(f"W-Recall: {results['W-Recall']:.3f}")
print(f"W-F1: {results['W-F1']:.3f}")
print(f"W-SEG: {results['W-SEG']:.3f}")
print(f"W-IoU: {results['W-IoU']:.3f}")

Processing 1694 GT cells and 1683 predicted cells...
  Confidence calculation done: 160.95s (1694 valid GT cells)
  Computing centroids and IoU for nearby cells...


KeyboardInterrupt: 

In [ ]:
print(f"W-F1: {results['W-F1']:.3f}")
print(f"W-Precision: {results['W-Precision']:.3f}")
print(f"W-Recall: {results['W-Recall']:.3f}")
print(f"W-IoU: {results['W-IoU']:.3f}")
print(f"W-SEG: {results['W-SEG']:.3f}")
print(f"Time = 220 seconds")

In [19]:
import numpy as np
from skimage.measure import regionprops
from tqdm import tqdm


def centroid_accuracy(ground_truth_mask, predicted_mask):
    ground_truth_mask = np.asarray(ground_truth_mask)
    predicted_mask = np.asarray(predicted_mask)
    if ground_truth_mask.shape != predicted_mask.shape:
        raise ValueError("Masks must share shape")

    gt_props = regionprops(ground_truth_mask)
    pred_props = regionprops(predicted_mask)

    # Precompute coordinate sets for ALL cells upfront
    gt_coords_sets = {prop.label: set(map(tuple, prop.coords)) for prop in gt_props}
    pred_coords_sets = {prop.label: set(map(tuple, prop.coords)) for prop in pred_props}

    hit_pred_labels = set()
    predicted_labels = np.array([])
    per_cell_iou = {}
    tp = fn = 0

    # Main loop with progress bar
    for prop in tqdm(gt_props, desc="Computing centroid accuracy", unit="cell"):
        gt_label = prop.label
        centroid = np.rint(prop.centroid).astype(int)
        centroid = tuple(np.clip(centroid, 0, np.array(ground_truth_mask.shape) - 1))
        pred_label = predicted_mask[centroid]
        predicted_labels = np.append(predicted_labels, pred_label)

        if pred_label > 0:
            tp += 1
            hit_pred_labels.add(int(pred_label))

            # Fast IoU using precomputed sets
            gt_set = gt_coords_sets[gt_label]
            pred_set = pred_coords_sets[pred_label]

            intersection = len(gt_set & pred_set)
            union = len(gt_set | pred_set)
            cell_iou = intersection / union if union > 0 else 0.0

            per_cell_iou[gt_label] = cell_iou
        else:
            fn += 1
            per_cell_iou[gt_label] = 0.0

    pred_labels = set(np.unique(predicted_mask))
    pred_labels.discard(0)
    fp = len(pred_labels - hit_pred_labels)
    fp_list = list(pred_labels - hit_pred_labels)
    merged_cells = len(predicted_labels) - len(np.unique(predicted_labels))

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0

    gt_binary = ground_truth_mask > 0
    pred_binary = predicted_mask > 0
    intersection = np.count_nonzero(gt_binary & pred_binary)
    dice = (
        (2 * intersection) / (gt_binary.sum() + pred_binary.sum())
        if (gt_binary.sum() + pred_binary.sum())
        else 0.0
    )

    # Calculate 3D IoU
    union = np.count_nonzero(gt_binary | pred_binary)
    iou_3d = intersection / union if union > 0 else 0.0

    return {
        "true_positives": tp,
        "false_negatives": fn,
        "false_positives": fp,
        "merged_cells": merged_cells,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "voxel_dice": dice,
        "iou_3d": iou_3d,
        "per_cell_iou": per_cell_iou,
        "false_positive_labels": fp_list,
    }

In [ ]:
import os
import tifffile

input_directory = r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Zebrafish_1"

image = tifffile.imread(os.path.join(input_directory, "data.tif"))

stitched_mask = tifffile.imread(
    os.path.join(input_directory, "3d_segmented_suijkerbuijk_drosophila_model.tif"),
)

cp4 = tifffile.imread(
    os.path.join(input_directory, "data_cp_masks.tif"),
)

ground_truth = tifffile.imread(
    os.path.join(input_directory, "GroundTruth.tif")
)

results_suijkerbuijk = centroid_accuracy(ground_truth, stitched_mask)
print(results_suijkerbuijk)
mean_iou_suijkerbuijk = np.mean(list(results_suijkerbuijk["per_cell_iou"].values()))
print(f"Mean IoU Suijkerbuijk: {mean_iou_suijkerbuijk:.3f}")

results_cp4 = centroid_accuracy(ground_truth, cp4)
print(results_cp4)
mean_iou_cp4 = np.mean(list(results_cp4["per_cell_iou"].values()))
print(f"Mean IoU CP4: {mean_iou_cp4:.3f}")

Computing centroid accuracy: 100%|██████████| 12793/12793 [00:05<00:00, 2383.09cell/s]


{'true_positives': 12376, 'false_negatives': 417, 'false_positives': 1360, 'merged_cells': 718, 'precision': 0.900990099009901, 'recall': 0.9674040490893457, 'f1': 0.9330166987070753, 'voxel_dice': 0.6847328458817838, 'iou_3d': 0.5206036231786262, 'per_cell_iou': {1: 0.5790527018012008, 2: 0.6065759637188208, 3: 0.38507847774672116, 4: 0.4475402699172834, 5: 0.0, 6: 0.5617726051924798, 7: 0.47619047619047616, 8: 0.5824104234527687, 9: 0.45702952842661965, 10: 0.5433596118859915, 11: 0.5547013601419278, 12: 0.5190905190905191, 13: 0.4256014256014256, 14: 0.584247258225324, 15: 0.4115025466893039, 16: 0.5481814773153356, 17: 0.468034727703236, 18: 0.3624511082138201, 19: 0.5390977443609023, 20: 0.10593434343434344, 21: 0.42330651440436023, 22: 0.46866791744840525, 23: 0.5578734858681023, 24: 0.5700389105058365, 25: 0.46203623813632444, 26: 0.5357905982905983, 27: 0.5074404761904762, 28: 0.414651493598862, 29: 0.516914749661705, 30: 0.2799145299145299, 31: 0.35561497326203206, 32: 0.55130

Computing centroid accuracy: 100%|██████████| 12793/12793 [00:04<00:00, 3078.77cell/s]


{'true_positives': 11350, 'false_negatives': 1443, 'false_positives': 1354, 'merged_cells': 2038, 'precision': 0.8934193954659949, 'recall': 0.8872039396544985, 'f1': 0.8903008197042789, 'voxel_dice': 0.5711498987531696, 'iou_3d': 0.3997269540414196, 'per_cell_iou': {1: 0.23710482529118138, 2: 0.4166168559473999, 3: 0.25673364685763145, 4: 0.0, 5: 0.18575667655786351, 6: 0.27456258411843876, 7: 0.08392390898918314, 8: 0.2526080476900149, 9: 0.019686800894854587, 10: 0.25142495250158325, 11: 0.4362017804154303, 12: 0.20499021526418787, 13: 0.0660321237358715, 14: 0.18805657412257726, 15: 0.19118279569892474, 16: 0.01809954751131222, 17: 0.0, 18: 0.0, 19: 0.3341875961045618, 20: 0.0, 21: 0.1763911399243652, 22: 0.07029015120555783, 23: 0.0, 24: 0.18495934959349594, 25: 0.20704467353951891, 26: 0.0, 27: 0.0, 28: 0.0, 29: 0.0, 30: 0.0, 31: 0.09558823529411764, 32: 0.042986425339366516, 33: 0.0, 34: 0.0, 35: 0.06429070580013976, 36: 0.35994677312042583, 37: 0.0, 38: 0.023476523476523476, 39

In [26]:
mean_iou_suijkerbuijk = np.mean(list(results_suijkerbuijk["per_cell_iou"].values()))
print(f"Mean IoU Suijkerbuijk: {mean_iou_suijkerbuijk:.3f}")
mean_iou_cp4 = np.mean(list(results_cp4["per_cell_iou"].values()))
print(f"Mean IoU CP4: {mean_iou_cp4:.3f}")

Mean IoU Suijkerbuijk: 0.602
Mean IoU CP4: 0.603


In [28]:
# Training fruitfly model

training_dir = r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Drosophila_1\model"


input_directory = r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Drosophila_1"
image = tifffile.imread(os.path.join(input_directory, f"data.tif"))
segmentation = tifffile.imread(os.path.join(input_directory, f"GroundTruth.tif"))
count = 0
# select 5 random integers between 0 and esg.shape[0]-1
selected_slices = np.random.choice(segmentation.shape[0], size=20, replace=False)
for slice in selected_slices:
    if count >= 7:
        break
    if len(np.unique(segmentation[slice, :, :])) < 5:
        continue
    count += 1
    tifffile.imwrite(
        os.path.join(training_dir, f"slice_{slice}_img.tif"),
        image[slice, :, :],
        imagej=True,
    )
    tifffile.imwrite(
        os.path.join(training_dir, f"slice_{slice}_mask.tif"),
        segmentation[slice, :, :],
        imagej=True,
    )

In [29]:
from cellpose import io, models, train

io.logger_setup()

training_dir = r"C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Drosophila_1\model"


output = io.load_train_test_data(
    training_dir,
    image_filter="_img",
    mask_filter="_mask",
    look_one_level_down=False,
)
images, labels, image_names, test_images, test_labels, image_names_test = output

model = models.CellposeModel(gpu=True)

model_path, train_losses, test_losses = train.train_seg(
    model.net,
    train_data=images,
    train_labels=labels,
    test_data=test_images,
    test_labels=test_labels,
    weight_decay=0.1,
    learning_rate=1e-5,
    n_epochs=100,
    save_path=training_dir,
    model_name="drosophila",
)

2026-02-04 14:10:45,500 [INFO] WRITING LOG OUTPUT TO C:\Users\6331823\.cellpose\run.log
2026-02-04 14:10:45,500 [INFO] 
cellpose version: 	4.0.7 
platform:       	win32 
python version: 	3.11.0 
torch version:  	2.7.1+cu128
2026-02-04 14:10:45,517 [INFO] not all flows are present, running flow generation for all images
2026-02-04 14:10:45,703 [INFO] 7 / 7 images in C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Drosophila_1\model folder have labels
2026-02-04 14:10:45,858 [INFO] ** TORCH CUDA version installed and working. **
2026-02-04 14:10:45,858 [INFO] >>>> using GPU (CUDA)
2026-02-04 14:10:46,789 [INFO] >>>> loading model C:\Users\6331823\.cellpose\models\cpsam
2026-02-04 14:10:47,451 [INFO] computing flows for labels


100%|██████████| 7/7 [00:00<00:00, 16.73it/s]

2026-02-04 14:10:47,870 [INFO] >>> computing diameters



100%|██████████| 7/7 [00:00<00:00, 638.53it/s]

2026-02-04 14:10:47,883 [INFO] >>> normalizing {'lowhigh': None, 'percentile': None, 'normalize': True, 'norm3D': True, 'sharpen_radius': 0, 'smooth_radius': 0, 'tile_norm_blocksize': 0, 'tile_norm_smooth3D': 1, 'invert': False}
2026-02-04 14:10:47,971 [INFO] >>> n_epochs=100, n_train=7, n_test=None
2026-02-04 14:10:47,972 [INFO] >>> AdamW, learning_rate=0.00001, weight_decay=0.10000
2026-02-04 14:10:47,973 [INFO] >>> saving model to C:\Users\6331823\Downloads\NIS3D\NIS3D\NIS3D\Drosophila_1\model\models\drosophila


2026-02-04 14:10:49,452 [INFO] 0, train_loss=0.2040, test_loss=0.0000, LR=0.000000, time 1.48s
2026-02-04 14:10:53,518 [INFO] 5, train_loss=0.1796, test_loss=0.0000, LR=0.000006, time 5.54s
2026-02-04 14:10:57,639 [INFO] 10, train_loss=0.2010, test_loss=0.0000, LR=0.000010, time 9.67s
2026-02-04 14:11:05,520 [INFO] 20, train_loss=0.1567, test_loss=0.0000, LR=0.000010, time 17.55s
2026-02-04 14:11:13,450 [INFO] 30, train_loss=0.1453, test_loss=0.0000, LR=0.000010, time 25.48s
2026-02-04 14:11:21,401 [INFO] 40, train_loss=0.1636, test_loss=0.0000, LR=0.000010, time 33.43s
2026-02-04 14:11:29,285 [INFO] 50, train_loss=0.1535, test_loss=0.0000, LR=0.000005, time 41.31s
2026-02-04 14:11:37,234 [INFO] 60, train_loss=0.1278, test_loss=0.0000, LR=0.000001, time 49.26s
2026-02-04 14:11:45,253 [INFO] 70, train_loss=0.1485, test_loss=0.0000, LR=0.000000, time 57.28s
2026-02-04 14:11:53,143 [INFO] 80, train_loss=0.1829, test_loss=0.0000, LR=0.000000, time 65.17s
2026-02-04 14:12:01,071 [INFO] 90, 